# DATA INTEGRITY & CLEANING

In [2]:
# ============================================================
# STEP 1: DATA INTEGRITY & CLEANING
# FA-AIPMTM: Adaptive Personalized Music Teaching
# ============================================================

import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------

BASE_PATH = r"F:\.0 Work\4358\Data"

CSV_PATH = os.path.join(
    BASE_PATH,
    "music_teaching_effect_dataset.csv"
)

GENRES = [
    "blues",
    "classical",
    "country",
    "disco",
    "hiphop",
    "jazz",
    "metal",
    "pop",
    "reggae",
    "rock"
]

# ------------------------------------------------------------
# 2. LOAD CSV DATASET
# ------------------------------------------------------------

df = pd.read_csv(CSV_PATH)

print("=" * 70)
print("DATASET LOADED")
print("=" * 70)

print(f"CSV path       : {CSV_PATH}")
print(f"Number records : {len(df)}")
print(f"Number columns : {len(df.columns)}")

print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:02d}. {col}")


# ------------------------------------------------------------
# 3. CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "Student_ID",
    "Practice_Hours",
    "Assignment_Score",
    "Attendance_Rate",
    "Pitch_Accuracy",
    "Rhythm_Accuracy",
    "Tempo_Stability",
    "MFCC_1",
    "MFCC_2",
    "MFCC_3",
    "MFCC_4",
    "MFCC_5",
    "MFCC_6",
    "MFCC_7",
    "MFCC_8",
    "MFCC_9",
    "MFCC_10",
    "Feedback_Sentiment",
    "Feedback_Length",
    "Teaching_Effectiveness"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    print("\nWARNING: Missing columns:")
    print(missing_columns)
else:
    print("\nAll required columns are present.")


# ------------------------------------------------------------
# 4. DATA TYPES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATA TYPES")
print("=" * 70)

print(df.dtypes)


# ------------------------------------------------------------
# 5. MISSING VALUE CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MISSING VALUE ANALYSIS")
print("=" * 70)

missing_values = df.isnull().sum()

missing_table = pd.DataFrame({
    "Column": missing_values.index,
    "Missing_Count": missing_values.values,
    "Missing_Percentage": (
        missing_values.values / len(df) * 100
    )
})

print(missing_table.to_string(index=False))

total_missing = df.isnull().sum().sum()

print(f"\nTotal missing values: {total_missing}")


# ------------------------------------------------------------
# 6. DUPLICATE RECORD CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DUPLICATE RECORD ANALYSIS")
print("=" * 70)

duplicate_rows = df.duplicated().sum()

print(f"Exact duplicate rows: {duplicate_rows}")


# ------------------------------------------------------------
# 7. STUDENT_ID UNIQUENESS CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STUDENT ID ANALYSIS")
print("=" * 70)

unique_students = df["Student_ID"].nunique()
total_records = len(df)

print(f"Total records       : {total_records}")
print(f"Unique Student IDs  : {unique_students}")
print(
    f"Repeated-ID records : "
    f"{total_records - unique_students}"
)

duplicate_student_ids = (
    df["Student_ID"]
    .value_counts()
)

duplicate_student_ids = duplicate_student_ids[
    duplicate_student_ids > 1
]

if len(duplicate_student_ids) > 0:

    print("\nStudent IDs occurring more than once:")
    print(duplicate_student_ids)

else:
    print("\nNo repeated Student IDs found.")


# ------------------------------------------------------------
# 8. TARGET VARIABLE ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TARGET VARIABLE ANALYSIS")
print("=" * 70)

print(
    df["Teaching_Effectiveness"]
    .value_counts(dropna=False)
)

target_distribution = (
    df["Teaching_Effectiveness"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTarget distribution (%):")
print(target_distribution)


# ------------------------------------------------------------
# 9. CHECK TARGET LABELS
# ------------------------------------------------------------

expected_labels = {
    "Low",
    "Medium",
    "High"
}

actual_labels = set(
    df["Teaching_Effectiveness"]
    .dropna()
    .astype(str)
    .unique()
)

print("\nExpected labels:", expected_labels)
print("Actual labels  :", actual_labels)

unexpected_labels = actual_labels - expected_labels

if unexpected_labels:
    print("\nWARNING: Unexpected target labels:")
    print(unexpected_labels)
else:
    print("\nTarget labels are Low / Medium / High.")


# ------------------------------------------------------------
# 10. NUMERICAL FEATURE RANGE CHECK
# ------------------------------------------------------------

numeric_features = [
    "Practice_Hours",
    "Assignment_Score",
    "Attendance_Rate",
    "Pitch_Accuracy",
    "Rhythm_Accuracy",
    "Tempo_Stability",
    "MFCC_1",
    "MFCC_2",
    "MFCC_3",
    "MFCC_4",
    "MFCC_5",
    "MFCC_6",
    "MFCC_7",
    "MFCC_8",
    "MFCC_9",
    "MFCC_10",
    "Feedback_Sentiment",
    "Feedback_Length"
]

print("\n" + "=" * 70)
print("NUMERICAL FEATURE SUMMARY")
print("=" * 70)

print(
    df[numeric_features]
    .describe()
    .T
    .round(4)
)


# ------------------------------------------------------------
# 11. INFINITE / INVALID VALUE CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INVALID NUMERICAL VALUE CHECK")
print("=" * 70)

inf_counts = np.isinf(
    df[numeric_features].select_dtypes(
        include=[np.number]
    )
).sum()

print("Infinite values per feature:")
print(inf_counts)

total_inf = inf_counts.sum()

print(f"\nTotal infinite values: {total_inf}")


# ------------------------------------------------------------
# 12. AUDIO FILE DISCOVERY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("AUDIO FILE DISCOVERY")
print("=" * 70)

audio_files = []

for genre in GENRES:

    genre_path = os.path.join(
        BASE_PATH,
        "genres_original",
        genre
    )

    if not os.path.exists(genre_path):
        print(f"WARNING: Folder not found: {genre_path}")
        continue

    files = [
        f for f in os.listdir(genre_path)
        if f.lower().endswith(".wav")
    ]

    print(f"{genre:12s}: {len(files):4d} WAV files")

    for file_name in files:

        audio_files.append({
            "Genre": genre,
            "File_Name": file_name,
            "File_Path": os.path.join(
                genre_path,
                file_name
            )
        })


audio_df = pd.DataFrame(audio_files)

print("\nTotal WAV files found:", len(audio_df))


# ------------------------------------------------------------
# 13. AUDIO FILE FORMAT CHECK
# ------------------------------------------------------------

if len(audio_df) > 0:

    print("\n" + "=" * 70)
    print("AUDIO FILE SUMMARY")
    print("=" * 70)

    print(
        audio_df.groupby("Genre")
        .size()
        .sort_index()
    )

    print("\nExample audio files:")
    print(
        audio_df.head(10)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 14. CHECK FOR CORRUPTED / MISSING AUDIO FILES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("AUDIO PATH VALIDATION")
print("=" * 70)

audio_df["Exists"] = audio_df["File_Path"].apply(
    os.path.exists
)

missing_audio = (~audio_df["Exists"]).sum()

print(f"Missing audio paths: {missing_audio}")

if missing_audio == 0:
    print("All discovered WAV paths are valid.")
else:
    print("\nMissing files:")
    print(
        audio_df[
            ~audio_df["Exists"]
        ]
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 15. EXTRACT POSSIBLE STUDENT ID FROM AUDIO FILE NAME
# ------------------------------------------------------------

def extract_student_id(filename):

    """
    Attempts to extract the numeric ID from
    filenames such as:

    S079.wav
    S146.wav
    S213.wav

    Returns the ID as a string.
    """

    name = os.path.splitext(filename)[0]

    if name.upper().startswith("S"):
        return name[1:]

    return name


if len(audio_df) > 0:

    audio_df["Audio_ID"] = (
        audio_df["File_Name"]
        .apply(extract_student_id)
        .astype(str)
    )

    df["Student_ID_String"] = (
        df["Student_ID"]
        .astype(str)
    )

    # Normalize IDs such as 79 -> 079 if required
    df["Student_ID_String"] = (
        df["Student_ID_String"]
        .str.replace(".0", "", regex=False)
        .str.zfill(3)
    )

    audio_df["Audio_ID"] = (
        audio_df["Audio_ID"]
        .str.replace(".0", "", regex=False)
        .str.zfill(3)
    )


# ------------------------------------------------------------
# 16. MATCH CSV RECORDS WITH AUDIO FILES
# ------------------------------------------------------------

if len(audio_df) > 0:

    csv_ids = set(
        df["Student_ID_String"]
    )

    audio_ids = set(
        audio_df["Audio_ID"]
    )

    matched_ids = csv_ids.intersection(
        audio_ids
    )

    csv_without_audio = csv_ids - audio_ids

    audio_without_csv = audio_ids - csv_ids

    print("\n" + "=" * 70)
    print("CSV–AUDIO ID MATCHING")
    print("=" * 70)

    print(f"Unique CSV IDs          : {len(csv_ids)}")
    print(f"Unique audio IDs        : {len(audio_ids)}")
    print(f"Matched IDs             : {len(matched_ids)}")
    print(f"CSV IDs without audio   : {len(csv_without_audio)}")
    print(f"Audio IDs without CSV   : {len(audio_without_csv)}")

    if csv_without_audio:
        print("\nCSV IDs without matching audio:")
        print(sorted(csv_without_audio))

    if audio_without_csv:
        print("\nAudio IDs without matching CSV:")
        print(sorted(audio_without_csv))


# ------------------------------------------------------------
# 17. GENRE DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GENRE DISTRIBUTION")
print("=" * 70)

genre_distribution = (
    audio_df["Genre"]
    .value_counts()
    .sort_index()
)

print(genre_distribution)


# ------------------------------------------------------------
# 18. CREATE CLEAN TABULAR DATASET
# ------------------------------------------------------------

clean_df = df.copy()

# Remove exact duplicates
clean_df = clean_df.drop_duplicates()

# Remove rows with missing target
clean_df = clean_df.dropna(
    subset=["Teaching_Effectiveness"]
)

# Replace infinite values
clean_df[numeric_features] = (
    clean_df[numeric_features]
    .replace([np.inf, -np.inf], np.nan)
)

print("\n" + "=" * 70)
print("CLEAN DATASET")
print("=" * 70)

print(f"Original records : {len(df)}")
print(f"Clean records    : {len(clean_df)}")
print(
    f"Removed records  : "
    f"{len(df) - len(clean_df)}"
)


# ------------------------------------------------------------
# 19. REMOVE IDENTIFIER FROM MODEL FEATURES
# ------------------------------------------------------------

if "Student_ID_String" in clean_df.columns:
    clean_df = clean_df.drop(
        columns=["Student_ID_String"]
    )

print("\nStudent_ID will NOT be used as a model feature.")


# ------------------------------------------------------------
# 20. SAVE CLEAN DATASET
# ------------------------------------------------------------

OUTPUT_PATH = os.path.join(
    BASE_PATH,
    "music_teaching_effect_dataset_clean.csv"
)

clean_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\n" + "=" * 70)
print("STEP 1 COMPLETED")
print("=" * 70)

print(f"Clean dataset saved to:")
print(OUTPUT_PATH)

print("\nFinal shape:")
print(clean_df.shape)

DATASET LOADED
CSV path       : F:\.0 Work\4358\Data\music_teaching_effect_dataset.csv
Number records : 999
Number columns : 20

Columns:
01. Student_ID
02. Practice_Hours
03. Assignment_Score
04. Attendance_Rate
05. Pitch_Accuracy
06. Rhythm_Accuracy
07. Tempo_Stability
08. MFCC_1
09. MFCC_2
10. MFCC_3
11. MFCC_4
12. MFCC_5
13. MFCC_6
14. MFCC_7
15. MFCC_8
16. MFCC_9
17. MFCC_10
18. Feedback_Sentiment
19. Feedback_Length
20. Teaching_Effectiveness

All required columns are present.

DATA TYPES
Student_ID                 object
Practice_Hours              int64
Assignment_Score            int64
Attendance_Rate           float64
Pitch_Accuracy            float64
Rhythm_Accuracy           float64
Tempo_Stability           float64
MFCC_1                    float64
MFCC_2                    float64
MFCC_3                    float64
MFCC_4                    float64
MFCC_5                    float64
MFCC_6                    float64
MFCC_7                    float64
MFCC_8                  

# AUDIO PREPROCESSING

In [3]:
# ============================================================
# STEP 2: AUDIO PREPROCESSING
# Spectral Subtraction-Based Noise Reduction
#
# FA-AIPMTM: Adaptive Personalized Music Teaching
# ============================================================

import os
import numpy as np
import librosa
import soundfile as sf
from tqdm import tqdm

# ------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------

BASE_PATH = r"F:\.0 Work\4358\Data"

INPUT_AUDIO_PATH = os.path.join(
    BASE_PATH,
    "genres_original"
)

OUTPUT_AUDIO_PATH = os.path.join(
    BASE_PATH,
    "genres_denoised"
)

GENRES = [
    "blues",
    "classical",
    "country",
    "disco",
    "hiphop",
    "jazz",
    "metal",
    "pop",
    "reggae",
    "rock"
]

# Create output directory
os.makedirs(OUTPUT_AUDIO_PATH, exist_ok=True)

for genre in GENRES:
    os.makedirs(
        os.path.join(OUTPUT_AUDIO_PATH, genre),
        exist_ok=True
    )


# ------------------------------------------------------------
# 2. AUDIO PARAMETERS
# ------------------------------------------------------------

TARGET_SR = 16000

N_FFT = 512
WIN_LENGTH = 400       # 25 ms at 16 kHz
HOP_LENGTH = 256       # 16 ms
WINDOW = "hann"

# Noise estimation duration
NOISE_DURATION = 0.5

# Spectral flooring factor
SPECTRAL_FLOOR = 0.002

# Oversubtraction factor
ALPHA = 1.0


# ------------------------------------------------------------
# 3. SPECTRAL SUBTRACTION FUNCTION
# ------------------------------------------------------------

def spectral_subtraction(
    audio,
    sr=TARGET_SR,
    n_fft=N_FFT,
    win_length=WIN_LENGTH,
    hop_length=HOP_LENGTH,
    noise_duration=NOISE_DURATION,
    spectral_floor=SPECTRAL_FLOOR,
    alpha=ALPHA
):
    """
    Spectral subtraction-based noise reduction.

    Input:
        audio : 1-D audio waveform
        sr    : sampling rate

    Output:
        enhanced_audio : denoised waveform
    """

    # --------------------------------------------------------
    # STFT
    # --------------------------------------------------------

    stft = librosa.stft(
        audio,
        n_fft=n_fft,
        win_length=win_length,
        hop_length=hop_length,
        window="hann"
    )

    magnitude = np.abs(stft)
    phase = np.angle(stft)

    power = magnitude ** 2

    # --------------------------------------------------------
    # Estimate noise from initial 0.5 seconds
    # --------------------------------------------------------

    noise_samples = int(
        noise_duration * sr
    )

    noise_samples = min(
        noise_samples,
        len(audio)
    )

    noise_audio = audio[:noise_samples]

    noise_stft = librosa.stft(
        noise_audio,
        n_fft=n_fft,
        win_length=win_length,
        hop_length=hop_length,
        window="hann"
    )

    noise_power = np.mean(
        np.abs(noise_stft) ** 2,
        axis=1,
        keepdims=True
    )

    # --------------------------------------------------------
    # Spectral subtraction
    # --------------------------------------------------------

    enhanced_power = (
        power -
        alpha * noise_power
    )

    # Prevent negative spectral power
    enhanced_power = np.maximum(
        enhanced_power,
        spectral_floor * power.max()
    )

    # Convert power to magnitude
    enhanced_magnitude = np.sqrt(
        enhanced_power
    )

    # --------------------------------------------------------
    # Reconstruct complex spectrum
    # --------------------------------------------------------

    enhanced_stft = (
        enhanced_magnitude *
        np.exp(1j * phase)
    )

    # --------------------------------------------------------
    # Inverse STFT
    # --------------------------------------------------------

    enhanced_audio = librosa.istft(
        enhanced_stft,
        hop_length=hop_length,
        win_length=win_length,
        window="hann",
        length=len(audio)
    )

    return enhanced_audio


# ------------------------------------------------------------
# 4. NORMALIZE AUDIO AMPLITUDE
# ------------------------------------------------------------

def normalize_audio(audio):
    """
    Peak-normalize audio to [-1, 1].
    """

    peak = np.max(
        np.abs(audio)
    )

    if peak > 0:
        audio = audio / peak

    return audio


# ------------------------------------------------------------
# 5. PROCESS ONE AUDIO FILE
# ------------------------------------------------------------

def process_audio_file(
    input_file,
    output_file
):

    try:

        # Load audio
        audio, sr = librosa.load(
            input_file,
            sr=TARGET_SR,
            mono=True
        )

        # Check empty file
        if len(audio) == 0:
            return False, "Empty audio"

        # Spectral subtraction
        denoised_audio = spectral_subtraction(
            audio,
            sr=sr
        )

        # Peak normalization
        denoised_audio = normalize_audio(
            denoised_audio
        )

        # Save
        sf.write(
            output_file,
            denoised_audio,
            TARGET_SR,
            subtype="PCM_16"
        )

        return True, "Success"

    except Exception as e:

        return False, str(e)


# ------------------------------------------------------------
# 6. PROCESS ALL GENRES
# ------------------------------------------------------------

processing_results = []

print("=" * 70)
print("STEP 2: SPECTRAL SUBTRACTION AUDIO PREPROCESSING")
print("=" * 70)

for genre in GENRES:

    input_folder = os.path.join(
        INPUT_AUDIO_PATH,
        genre
    )

    output_folder = os.path.join(
        OUTPUT_AUDIO_PATH,
        genre
    )

    if not os.path.exists(input_folder):

        print(
            f"\nWARNING: Folder not found: "
            f"{input_folder}"
        )

        continue

    wav_files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith(".wav")
    ]

    print(
        f"\nProcessing {genre}: "
        f"{len(wav_files)} files"
    )

    success_count = 0
    failed_count = 0

    for filename in tqdm(
        wav_files,
        desc=genre
    ):

        input_file = os.path.join(
            input_folder,
            filename
        )

        output_file = os.path.join(
            output_folder,
            filename
        )

        success, message = process_audio_file(
            input_file,
            output_file
        )

        processing_results.append({
            "Genre": genre,
            "File_Name": filename,
            "Input_Path": input_file,
            "Output_Path": output_file,
            "Status": message
        })

        if success:
            success_count += 1
        else:
            failed_count += 1

    print(
        f"Completed {genre}: "
        f"{success_count} successful, "
        f"{failed_count} failed"
    )


# ------------------------------------------------------------
# 7. PROCESSING SUMMARY
# ------------------------------------------------------------

import pandas as pd

results_df = pd.DataFrame(
    processing_results
)

successful = (
    results_df["Status"] == "Success"
).sum()

failed = (
    results_df["Status"] != "Success"
).sum()

print("\n" + "=" * 70)
print("STEP 2 COMPLETED")
print("=" * 70)

print(
    f"Total audio files : {len(results_df)}"
)

print(
    f"Successful        : {successful}"
)

print(
    f"Failed            : {failed}"
)

print(
    f"Output directory  : {OUTPUT_AUDIO_PATH}"
)


# ------------------------------------------------------------
# 8. SAVE PROCESSING LOG
# ------------------------------------------------------------

LOG_PATH = os.path.join(
    BASE_PATH,
    "audio_preprocessing_log.csv"
)

results_df.to_csv(
    LOG_PATH,
    index=False
)

print(
    f"\nProcessing log saved to:\n{LOG_PATH}"
)


# ------------------------------------------------------------
# 9. GENRE-WISE SUMMARY
# ------------------------------------------------------------

print("\nGenre-wise processing summary:")

summary = (
    results_df
    .groupby(["Genre", "Status"])
    .size()
    .unstack(fill_value=0)
)

print(summary)


# ------------------------------------------------------------
# 10. VERIFY OUTPUT FILE COUNT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OUTPUT AUDIO VERIFICATION")
print("=" * 70)

total_output_files = 0

for genre in GENRES:

    folder = os.path.join(
        OUTPUT_AUDIO_PATH,
        genre
    )

    if os.path.exists(folder):

        count = len([
            f for f in os.listdir(folder)
            if f.lower().endswith(".wav")
        ])

        total_output_files += count

        print(
            f"{genre:12s}: {count:4d} files"
        )

print(
    f"\nTotal denoised WAV files: "
    f"{total_output_files}"
)

STEP 2: SPECTRAL SUBTRACTION AUDIO PREPROCESSING

Processing blues: 100 files


blues: 100%|█████████████████████████████████████████████████████████████████████████| 100/100 [00:28<00:00,  3.51it/s]


Completed blues: 100 successful, 0 failed

Processing classical: 100 files


classical: 100%|█████████████████████████████████████████████████████████████████████| 100/100 [00:14<00:00,  7.00it/s]


Completed classical: 100 successful, 0 failed

Processing country: 100 files


...

Completed pop: 100 successful, 0 failed

Processing reggae: 100 files


reggae: 100%|████████████████████████████████████████████████████████████████████████| 100/100 [00:24<00:00,  4.16it/s]


Completed reggae: 100 successful, 0 failed

Processing rock: 100 files


rock: 100%|██████████████████████████████████████████████████████████████████████████| 100/100 [00:27<00:00,  3.62it/s]

Completed rock: 100 successful, 0 failed

STEP 2 COMPLETED
Total audio files : 1000
Successful        : 999
Failed            : 1
Output directory  : F:\.0 Work\4358\Data\genres_denoised

Processing log saved to:
F:\.0 Work\4358\Data\audio_preprocessing_log.csv

Genre-wise processing summary:
Status        Success
Genre                
blues      0      100
classical  0      100
country    0      100
disco      0      100
hiphop     0      100
jazz       1       99
metal      0      100
pop        0      100
reggae     0      100
rock       0      100

OUTPUT AUDIO VERIFICATION
blues       :  100 files
classical   :  100 files
country     :  100 files
disco       :  100 files
hiphop      :  100 files
jazz        :   99 files
metal       :  100 files
pop         :  100 files
reggae      :  100 files
rock        :  100 files

Total denoised WAV files: 999


# MFCC FEATURE EXTRACTION

In [4]:
# ============================================================
# STEP 3: MFCC FEATURE EXTRACTION
# FA-AIPMTM: Adaptive Personalized Music Teaching
# ============================================================

import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

# ------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------

BASE_PATH = r"F:\.0 Work\4358\Data"

INPUT_AUDIO_PATH = os.path.join(
    BASE_PATH,
    "genres_denoised"
)

OUTPUT_FEATURE_PATH = os.path.join(
    BASE_PATH,
    "mfcc_features"
)

os.makedirs(
    OUTPUT_FEATURE_PATH,
    exist_ok=True
)

GENRES = [
    "blues",
    "classical",
    "country",
    "disco",
    "hiphop",
    "jazz",
    "metal",
    "pop",
    "reggae",
    "rock"
]


# ------------------------------------------------------------
# 2. MFCC PARAMETERS
# ------------------------------------------------------------

SAMPLE_RATE = 16000

N_MFCC = 13

N_FFT = 512

WIN_LENGTH = 400       # 25 ms at 16 kHz

HOP_LENGTH = 160       # 10 ms at 16 kHz

N_MELS = 26

POWER = 2.0

LIFTER = 22


# ------------------------------------------------------------
# 3. MFCC EXTRACTION FUNCTION
# ------------------------------------------------------------

def extract_mfcc_features(audio_path):

    try:

        # ----------------------------------------------------
        # Load denoised audio
        # ----------------------------------------------------

        audio, sr = librosa.load(
            audio_path,
            sr=SAMPLE_RATE,
            mono=True
        )

        # ----------------------------------------------------
        # Check audio
        # ----------------------------------------------------

        if len(audio) == 0:
            raise ValueError("Empty audio file")

        # ----------------------------------------------------
        # MFCC extraction
        # ----------------------------------------------------

        mfcc = librosa.feature.mfcc(
            y=audio,
            sr=sr,
            n_mfcc=N_MFCC,
            n_fft=N_FFT,
            win_length=WIN_LENGTH,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
            power=POWER,
            lifter=LIFTER
        )

        # ----------------------------------------------------
        # MFCC matrix dimensions
        # ----------------------------------------------------

        # Shape:
        # 13 MFCC coefficients × number of frames
        #
        # Example:
        # (13, 1001)
        # ----------------------------------------------------

        # ----------------------------------------------------
        # Statistical aggregation
        # ----------------------------------------------------

        mfcc_mean = np.mean(
            mfcc,
            axis=1
        )

        mfcc_std = np.std(
            mfcc,
            axis=1
        )

        mfcc_min = np.min(
            mfcc,
            axis=1
        )

        mfcc_max = np.max(
            mfcc,
            axis=1
        )

        # ----------------------------------------------------
        # Combine all statistics
        # ----------------------------------------------------

        feature_vector = np.concatenate([
            mfcc_mean,
            mfcc_std,
            mfcc_min,
            mfcc_max
        ])

        return feature_vector, mfcc.shape

    except Exception as e:

        return None, str(e)


# ------------------------------------------------------------
# 4. FEATURE COLUMN NAMES
# ------------------------------------------------------------

feature_names = []

for i in range(1, N_MFCC + 1):
    feature_names.append(
        f"MFCC_{i}_Mean"
    )

for i in range(1, N_MFCC + 1):
    feature_names.append(
        f"MFCC_{i}_Std"
    )

for i in range(1, N_MFCC + 1):
    feature_names.append(
        f"MFCC_{i}_Min"
    )

for i in range(1, N_MFCC + 1):
    feature_names.append(
        f"MFCC_{i}_Max"
    )


# ------------------------------------------------------------
# 5. EXTRACT FEATURES FROM ALL GENRES
# ------------------------------------------------------------

all_features = []

failed_files = []

print("=" * 70)
print("STEP 3: MFCC FEATURE EXTRACTION")
print("=" * 70)

for genre in GENRES:

    genre_folder = os.path.join(
        INPUT_AUDIO_PATH,
        genre
    )

    if not os.path.exists(genre_folder):

        print(
            f"\nWARNING: Folder not found: "
            f"{genre_folder}"
        )

        continue

    wav_files = [
        f for f in os.listdir(genre_folder)
        if f.lower().endswith(".wav")
    ]

    print(
        f"\n{genre.upper()}: "
        f"{len(wav_files)} audio files"
    )

    for filename in tqdm(
        wav_files,
        desc=f"Extracting {genre}"
    ):

        audio_path = os.path.join(
            genre_folder,
            filename
        )

        features, shape = extract_mfcc_features(
            audio_path
        )

        # ----------------------------------------------------
        # Successful extraction
        # ----------------------------------------------------

        if features is not None:

            row = {
                "Genre": genre,
                "File_Name": filename,
                "Audio_Path": audio_path,
                "MFCC_Frame_Count": shape[1]
            }

            for name, value in zip(
                feature_names,
                features
            ):
                row[name] = value

            all_features.append(row)

        # ----------------------------------------------------
        # Failed extraction
        # ----------------------------------------------------

        else:

            failed_files.append({
                "Genre": genre,
                "File_Name": filename,
                "Reason": shape
            })


# ------------------------------------------------------------
# 6. CREATE MFCC DATAFRAME
# ------------------------------------------------------------

mfcc_df = pd.DataFrame(
    all_features
)

print("\n" + "=" * 70)
print("MFCC EXTRACTION SUMMARY")
print("=" * 70)

print(
    f"Successfully processed: "
    f"{len(mfcc_df)} files"
)

print(
    f"Failed files: "
    f"{len(failed_files)}"
)

print(
    f"Number of extracted features: "
    f"{len(feature_names)}"
)


# ------------------------------------------------------------
# 7. DISPLAY FEATURE SHAPE
# ------------------------------------------------------------

if not mfcc_df.empty:

    print("\nMFCC feature matrix shape:")

    print(
        mfcc_df.shape
    )

    print("\nFirst five records:")

    print(
        mfcc_df.head()
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 8. CHECK FOR MISSING MFCC VALUES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MFCC MISSING VALUE CHECK")
print("=" * 70)

if not mfcc_df.empty:

    missing_count = (
        mfcc_df[feature_names]
        .isnull()
        .sum()
        .sum()
    )

    print(
        f"Total missing MFCC values: "
        f"{missing_count}"
    )


# ------------------------------------------------------------
# 9. CHECK FOR INFINITE VALUES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MFCC INFINITE VALUE CHECK")
print("=" * 70)

if not mfcc_df.empty:

    infinite_count = np.isinf(
        mfcc_df[feature_names]
        .values
    ).sum()

    print(
        f"Total infinite MFCC values: "
        f"{infinite_count}"
    )


# ------------------------------------------------------------
# 10. SAVE MFCC FEATURES
# ------------------------------------------------------------

MFCC_OUTPUT = os.path.join(
    BASE_PATH,
    "mfcc_extracted_features.csv"
)

mfcc_df.to_csv(
    MFCC_OUTPUT,
    index=False
)

print("\n" + "=" * 70)
print("MFCC FEATURE FILE SAVED")
print("=" * 70)

print(MFCC_OUTPUT)


# ------------------------------------------------------------
# 11. SAVE FAILED FILE LOG
# ------------------------------------------------------------

FAILED_OUTPUT = os.path.join(
    BASE_PATH,
    "mfcc_failed_files.csv"
)

failed_df = pd.DataFrame(
    failed_files
)

failed_df.to_csv(
    FAILED_OUTPUT,
    index=False
)

print(
    f"\nFailed-file log saved to:"
    f"\n{FAILED_OUTPUT}"
)


# ------------------------------------------------------------
# 12. GENRE-WISE MFCC EXTRACTION SUMMARY
# ------------------------------------------------------------

if not mfcc_df.empty:

    print("\n" + "=" * 70)
    print("GENRE-WISE EXTRACTION")
    print("=" * 70)

    genre_summary = (
        mfcc_df
        .groupby("Genre")
        .size()
        .sort_index()
    )

    print(
        genre_summary
    )


# ------------------------------------------------------------
# 13. DISPLAY MFCC STATISTICS
# ------------------------------------------------------------

if not mfcc_df.empty:

    print("\n" + "=" * 70)
    print("MFCC STATISTICAL SUMMARY")
    print("=" * 70)

    print(
        mfcc_df[feature_names]
        .describe()
        .T
        .round(4)
    )


# ------------------------------------------------------------
# 14. FINAL MESSAGE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 3 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    "\nThe extracted MFCC representation contains:"
)

print(
    f"• {N_MFCC} MFCC coefficients"
)

print(
    "• Mean statistics"
)

print(
    "• Standard deviation statistics"
)

print(
    "• Minimum statistics"
)

print(
    "• Maximum statistics"
)

print(
    f"• Total audio features per file: "
    f"{N_MFCC * 4}"
)

print(
    "\nNext stage: "
    "Step 4 - Tabular + MFCC Feature Fusion and "
    "Training-Set Min-Max Normalization"
)

STEP 3: MFCC FEATURE EXTRACTION

BLUES: 100 audio files


Extracting blues: 100%|██████████████████████████████████████████████████████████████| 100/100 [00:05<00:00, 17.75it/s]



CLASSICAL: 100 audio files


Extracting classical: 100%|██████████████████████████████████████████████████████████| 100/100 [00:05<00:00, 19.28it/s]



COUNTRY: 100 audio files


...


REGGAE: 100 audio files


Extracting reggae: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:06<00:00, 14.93it/s]



ROCK: 100 audio files


Extracting rock: 100%|███████████████████████████████████████████████████████████████| 100/100 [00:08<00:00, 11.12it/s]



MFCC EXTRACTION SUMMARY
Successfully processed: 999 files
Failed files: 0
Number of extracted features: 52

MFCC feature matrix shape:
(999, 56)

First five records:
Genre       File_Name                                                 Audio_Path  MFCC_Frame_Count  MFCC_1_Mean  MFCC_2_Mean  MFCC_3_Mean  MFCC_4_Mean  MFCC_5_Mean  MFCC_6_Mean  MFCC_7_Mean  MFCC_8_Mean  MFCC_9_Mean  MFCC_10_Mean  MFCC_11_Mean  MFCC_12_Mean  MFCC_13_Mean  MFCC_1_Std  MFCC_2_Std  MFCC_3_Std  MFCC_4_Std  MFCC_5_Std  MFCC_6_Std  MFCC_7_Std  MFCC_8_Std  MFCC_9_Std  MFCC_10_Std  MFCC_11_Std  MFCC_12_Std  MFCC_13_Std   MFCC_1_Min  MFCC_2_Min  MFCC_3_Min  MFCC_4_Min  MFCC_5_Min  MFCC_6_Min  MFCC_7_Min  MFCC_8_Min  MFCC_9_Min  MFCC_10_Min  MFCC_11_Min  MFCC_12_Min  MFCC_13_Min  MFCC_1_Max  MFCC_2_Max  MFCC_3_Max  MFCC_4_Max  MFCC_5_Max  MFCC_6_Max  MFCC_7_Max  MFCC_8_Max  MFCC_9_Max  MFCC_10_Max  MFCC_11_Max  MFCC_12_Max  MFCC_13_Max
blues blues.00000.wav F:\.0 Work\4358\Data\genres_denoised\blues\blues.00000.wav

# DATASET INTEGRATION, FEATURE PREPARATION & TRAIN/TEST SPLIT

In [7]:
# ============================================================
# STEP 4 - CORRECTED VERSION
# DATASET + AUDIO MFCC INTEGRATION + NORMALIZATION
# ============================================================

import os
import re
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

BASE_PATH = r"F:\.0 Work\4358\Data"

CSV_PATH = os.path.join(
    BASE_PATH,
    "genres_original",
    "music_teaching_effect_dataset.csv"
)

MFCC_PATH = os.path.join(
    BASE_PATH,
    "mfcc_extracted_features.csv"
)

OUTPUT_PATH = os.path.join(
    BASE_PATH,
    "step4_output"
)

os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)

print("=" * 70)
print("STEP 4: DATASET INTEGRATION AND FEATURE PREPARATION")
print("=" * 70)


# ------------------------------------------------------------
# 2. LOAD ORIGINAL CSV
# ------------------------------------------------------------

print("\n[1] Loading original dataset...")

df = pd.read_csv(CSV_PATH)

df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
)

print("Dataset shape:", df.shape)

print("\nTarget distribution:")
print(
    df["Teaching_Effectiveness"]
    .value_counts()
)


# ------------------------------------------------------------
# 3. LOAD AUDIO MFCC FEATURES
# ------------------------------------------------------------

print("\n[2] Loading extracted MFCC features...")

mfcc_df = pd.read_csv(MFCC_PATH)

print(
    "MFCC dataset shape:",
    mfcc_df.shape
)

print("\nMFCC columns:")
print(mfcc_df.columns.tolist())


# ------------------------------------------------------------
# 4. EXTRACT STUDENT_ID FROM FILE_NAME
# ------------------------------------------------------------

print("\n[3] Extracting Student_ID from audio filenames...")


def extract_student_id(filename):

    filename = str(filename)

    # Example:
    # S079.wav
    # S146.wav
    # S001.wav

    match = re.search(
        r"(S\d+)",
        filename,
        re.IGNORECASE
    )

    if match:
        return match.group(1).upper()

    return np.nan


mfcc_df["Student_ID"] = (
    mfcc_df["File_Name"]
    .apply(extract_student_id)
)


print("\nExtracted Student_ID examples:")

print(
    mfcc_df[
        ["File_Name", "Student_ID"]
    ].head(15)
)


# ------------------------------------------------------------
# 5. CHECK EXTRACTION
# ------------------------------------------------------------

missing_ids = (
    mfcc_df["Student_ID"]
    .isna()
    .sum()
)

print(
    "\nMFCC records without Student_ID:",
    missing_ids
)

if missing_ids > 0:

    print("\nFiles without valid Student_ID:")

    print(
        mfcc_df.loc[
            mfcc_df["Student_ID"].isna(),
            "File_Name"
        ].head(20)
    )


# ------------------------------------------------------------
# 6. CHECK DUPLICATE STUDENT IDS
# ------------------------------------------------------------

duplicate_ids = (
    mfcc_df["Student_ID"]
    .value_counts()
)

duplicate_ids = duplicate_ids[
    duplicate_ids > 1
]

print(
    "\nDuplicate Student_IDs in MFCC data:",
    len(duplicate_ids)
)

if len(duplicate_ids) > 0:

    print(
        duplicate_ids.head(20)
    )


# ------------------------------------------------------------
# 7. REMOVE MFCC RECORDS WITHOUT ID
# ------------------------------------------------------------

mfcc_df = mfcc_df.dropna(
    subset=["Student_ID"]
).copy()


# ------------------------------------------------------------
# 8. CHECK ORIGINAL CSV IDs
# ------------------------------------------------------------

print("\n[4] Checking Student_ID matching...")

csv_ids = set(
    df["Student_ID"]
    .astype(str)
    .str.upper()
)

mfcc_ids = set(
    mfcc_df["Student_ID"]
    .astype(str)
    .str.upper()
)

matched_ids = (
    csv_ids.intersection(mfcc_ids)
)

csv_only_ids = (
    csv_ids - mfcc_ids
)

mfcc_only_ids = (
    mfcc_ids - csv_ids
)

print(
    "CSV Student IDs:",
    len(csv_ids)
)

print(
    "MFCC Student IDs:",
    len(mfcc_ids)
)

print(
    "Matched IDs:",
    len(matched_ids)
)

print(
    "CSV IDs without MFCC:",
    len(csv_only_ids)
)

print(
    "MFCC IDs without CSV:",
    len(mfcc_only_ids)
)


# ------------------------------------------------------------
# 9. DISPLAY UNMATCHED RECORDS
# ------------------------------------------------------------

if len(csv_only_ids) > 0:

    print(
        "\nCSV records without corresponding MFCC:"
    )

    print(
        list(csv_only_ids)[:20]
    )


if len(mfcc_only_ids) > 0:

    print(
        "\nMFCC records without corresponding CSV:"
    )

    print(
        list(mfcc_only_ids)[:20]
    )


# ------------------------------------------------------------
# 10. MERGE DATASET WITH MFCC FEATURES
# ------------------------------------------------------------

print("\n[5] Merging CSV and audio MFCC data...")

# Rename audio-specific columns to avoid confusion

mfcc_merge = mfcc_df.copy()

# Add prefix to MFCC features generated from audio

audio_mfcc_columns = [
    col
    for col in mfcc_merge.columns
    if col.startswith("MFCC_")
]

rename_dict = {}

for col in audio_mfcc_columns:

    rename_dict[col] = (
        "Audio_" + col
    )

mfcc_merge = mfcc_merge.rename(
    columns=rename_dict
)


# Merge using Student_ID

merged_df = pd.merge(
    df,
    mfcc_merge,
    on="Student_ID",
    how="left",
    suffixes=("", "_audio")
)

print(
    "\nMerged dataset shape:",
    merged_df.shape
)


# ------------------------------------------------------------
# 11. CHECK AUDIO MFCC MATCHING
# ------------------------------------------------------------

audio_mfcc_feature_columns = [
    col
    for col in merged_df.columns
    if col.startswith("Audio_MFCC_")
]

print(
    "\nNumber of audio MFCC features:",
    len(audio_mfcc_feature_columns)
)

if len(audio_mfcc_feature_columns) > 0:

    first_audio_feature = (
        audio_mfcc_feature_columns[0]
    )

    matched_mfcc = (
        merged_df[first_audio_feature]
        .notna()
        .sum()
    )

    unmatched_mfcc = (
        merged_df[first_audio_feature]
        .isna()
        .sum()
    )

    print(
        "Records with audio MFCC:",
        matched_mfcc
    )

    print(
        "Records without audio MFCC:",
        unmatched_mfcc
    )


# ------------------------------------------------------------
# 12. CHECK AUDIO PATH
# ------------------------------------------------------------

if "Audio_Path" in merged_df.columns:

    print(
        "\nAudio path matching:"
    )

    print(
        "Available:",
        merged_df["Audio_Path"]
        .notna()
        .sum()
    )

    print(
        "Missing:",
        merged_df["Audio_Path"]
        .isna()
        .sum()
    )


# ------------------------------------------------------------
# 13. DEFINE FEATURE GROUPS
# ------------------------------------------------------------

behavioral_features = [
    "Practice_Hours",
    "Assignment_Score",
    "Attendance_Rate"
]

musical_features = [
    "Pitch_Accuracy",
    "Rhythm_Accuracy",
    "Tempo_Stability"
]

feedback_features = [
    "Feedback_Sentiment",
    "Feedback_Length"
]

# MFCC columns already present in original CSV

dataset_mfcc_features = [
    col
    for col in merged_df.columns
    if re.fullmatch(
        r"MFCC_\d+",
        col
    )
]


# ------------------------------------------------------------
# 14. AUDIO MFCC FEATURE GROUP
# ------------------------------------------------------------

audio_mfcc_features = [
    col
    for col in merged_df.columns
    if col.startswith(
        "Audio_MFCC_"
    )
]


print("\nFeature groups:")

print(
    "\nBehavioral features:",
    len(behavioral_features)
)

print(
    behavioral_features
)

print(
    "\nMusical features:",
    len(musical_features)
)

print(
    musical_features
)

print(
    "\nFeedback features:",
    len(feedback_features)
)

print(
    feedback_features
)

print(
    "\nOriginal dataset MFCC features:",
    len(dataset_mfcc_features)
)

print(
    dataset_mfcc_features
)

print(
    "\nAudio-derived MFCC features:",
    len(audio_mfcc_features)
)


# ------------------------------------------------------------
# 15. INITIAL MODEL FEATURE SET
# ------------------------------------------------------------

# We retain the original 10 MFCC columns as the main
# tabular MFCC feature group.
#
# The 52 audio-derived MFCC statistics are retained in the
# integrated dataset and can be tested separately or added
# during multimodal fusion.

model_features = (
    behavioral_features
    + musical_features
    + dataset_mfcc_features
    + feedback_features
)

print(
    "\nTotal initial model features:",
    len(model_features)
)

print(
    "\nModel features:"
)

for i, feature in enumerate(
    model_features,
    start=1
):

    print(
        f"{i:02d}. {feature}"
    )


# ------------------------------------------------------------
# 16. CONVERT FEATURES TO NUMERIC
# ------------------------------------------------------------

for col in model_features:

    merged_df[col] = pd.to_numeric(
        merged_df[col],
        errors="coerce"
    )


# ------------------------------------------------------------
# 17. CHECK MISSING VALUES
# ------------------------------------------------------------

print(
    "\n[6] Checking feature missing values..."
)

missing_features = (
    merged_df[
        model_features
    ]
    .isnull()
    .sum()
)

print(
    missing_features[
        missing_features > 0
    ]
)


# ------------------------------------------------------------
# 18. MEDIAN IMPUTATION
# ------------------------------------------------------------

for col in model_features:

    if merged_df[col].isnull().any():

        median_value = (
            merged_df[col]
            .median()
        )

        merged_df[col] = (
            merged_df[col]
            .fillna(median_value)
        )


print(
    "\nRemaining missing feature values:",
    merged_df[
        model_features
    ]
    .isnull()
    .sum()
    .sum()
)


# ------------------------------------------------------------
# 19. DEFINE X AND y
# ------------------------------------------------------------

X = merged_df[
    model_features
].copy()

y = merged_df[
    "Teaching_Effectiveness"
].copy()


# ------------------------------------------------------------
# 20. STRATIFIED 80:20 SPLIT
# ------------------------------------------------------------

print(
    "\n[7] Creating stratified 80:20 train/test split..."
)

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)


print(
    "\nTraining samples:",
    len(X_train)
)

print(
    "Testing samples:",
    len(X_test)
)


# ------------------------------------------------------------
# 21. CLASS DISTRIBUTION
# ------------------------------------------------------------

print(
    "\nTraining distribution:"
)

print(
    y_train.value_counts()
)

print(
    "\nTesting distribution:"
)

print(
    y_test.value_counts()
)


# ------------------------------------------------------------
# 22. MIN-MAX NORMALIZATION
# ------------------------------------------------------------

print(
    "\n[8] Applying Min-Max normalization..."
)

scaler = MinMaxScaler(
    feature_range=(0, 1)
)

# FIT ONLY ON TRAINING DATA

X_train_scaled = (
    scaler.fit_transform(
        X_train
    )
)

# TRANSFORM TEST USING TRAINING SCALER

X_test_scaled = (
    scaler.transform(
        X_test
    )
)


# ------------------------------------------------------------
# 23. CONVERT TO DATAFRAME
# ------------------------------------------------------------

X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=model_features,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=model_features,
    index=X_test.index
)


# ------------------------------------------------------------
# 24. ADD TARGET
# ------------------------------------------------------------

train_df = X_train_scaled.copy()

train_df[
    "Teaching_Effectiveness"
] = y_train.values


test_df = X_test_scaled.copy()

test_df[
    "Teaching_Effectiveness"
] = y_test.values


# ------------------------------------------------------------
# 25. SAVE TRAINING DATA
# ------------------------------------------------------------

TRAIN_PATH = os.path.join(
    OUTPUT_PATH,
    "train_dataset_scaled.csv"
)

train_df.to_csv(
    TRAIN_PATH,
    index=False
)


# ------------------------------------------------------------
# 26. SAVE TEST DATA
# ------------------------------------------------------------

TEST_PATH = os.path.join(
    OUTPUT_PATH,
    "test_dataset_scaled.csv"
)

test_df.to_csv(
    TEST_PATH,
    index=False
)


# ------------------------------------------------------------
# 27. SAVE INTEGRATED DATASET
# ------------------------------------------------------------

INTEGRATED_PATH = os.path.join(
    OUTPUT_PATH,
    "integrated_music_teaching_dataset.csv"
)

merged_df.to_csv(
    INTEGRATED_PATH,
    index=False
)


# ------------------------------------------------------------
# 28. SAVE SCALER
# ------------------------------------------------------------

SCALER_PATH = os.path.join(
    OUTPUT_PATH,
    "minmax_scaler.pkl"
)

joblib.dump(
    scaler,
    SCALER_PATH
)


# ------------------------------------------------------------
# 29. SAVE FEATURE INFORMATION
# ------------------------------------------------------------

feature_groups = []

for feature in model_features:

    if feature in behavioral_features:

        group = "Behavioral"

    elif feature in musical_features:

        group = "Musical"

    elif feature in dataset_mfcc_features:

        group = "MFCC"

    elif feature in feedback_features:

        group = "Feedback"

    else:

        group = "Other"

    feature_groups.append(group)


feature_info = pd.DataFrame({
    "Feature": model_features,
    "Feature_Group": feature_groups
})

FEATURE_INFO_PATH = os.path.join(
    OUTPUT_PATH,
    "feature_information.csv"
)

feature_info.to_csv(
    FEATURE_INFO_PATH,
    index=False
)


# ------------------------------------------------------------
# 30. SAVE AUDIO-MFCC DATA SEPARATELY
# ------------------------------------------------------------

AUDIO_MFCC_PATH = os.path.join(
    OUTPUT_PATH,
    "audio_mfcc_features_integrated.csv"
)

audio_export_columns = [
    "Student_ID"
]

if "Genre" in merged_df.columns:

    audio_export_columns.append(
        "Genre"
    )

if "File_Name" in merged_df.columns:

    audio_export_columns.append(
        "File_Name"
    )

audio_export_columns += (
    audio_mfcc_features
)

audio_export_columns = [
    col
    for col in audio_export_columns
    if col in merged_df.columns
]

merged_df[
    audio_export_columns
].to_csv(
    AUDIO_MFCC_PATH,
    index=False
)


# ------------------------------------------------------------
# 31. FINAL SUMMARY
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("STEP 4 COMPLETED")
print("=" * 70)

print(
    "\nOriginal records:",
    len(df)
)

print(
    "Integrated records:",
    len(merged_df)
)

print(
    "Training records:",
    len(train_df)
)

print(
    "Testing records:",
    len(test_df)
)

print(
    "Model features:",
    len(model_features)
)

print(
    "Audio MFCC features:",
    len(audio_mfcc_features)
)

print("\nSaved files:")

print(
    "\nIntegrated dataset:"
)

print(INTEGRATED_PATH)

print(
    "\nTraining dataset:"
)

print(TRAIN_PATH)

print(
    "\nTesting dataset:"
)

print(TEST_PATH)

print(
    "\nScaler:"
)

print(SCALER_PATH)

print(
    "\nFeature information:"
)

print(FEATURE_INFO_PATH)

print(
    "\nAudio MFCC:"
)

print(AUDIO_MFCC_PATH)

STEP 4: DATASET INTEGRATION AND FEATURE PREPARATION

[1] Loading original dataset...
Dataset shape: (999, 20)

Target distribution:
Teaching_Effectiveness
High      357
Low       326
Medium    316
Name: count, dtype: int64

[2] Loading extracted MFCC features...
MFCC dataset shape: (999, 56)

MFCC columns:
['Genre', 'File_Name', 'Audio_Path', 'MFCC_Frame_Count', 'MFCC_1_Mean', 'MFCC_2_Mean', 'MFCC_3_Mean', 'MFCC_4_Mean', 'MFCC_5_Mean', 'MFCC_6_Mean', 'MFCC_7_Mean', 'MFCC_8_Mean', 'MFCC_9_Mean', 'MFCC_10_Mean', 'MFCC_11_Mean', 'MFCC_12_Mean', 'MFCC_13_Mean', 'MFCC_1_Std', 'MFCC_2_Std', 'MFCC_3_Std', 'MFCC_4_Std', 'MFCC_5_Std', 'MFCC_6_Std', 'MFCC_7_Std', 'MFCC_8_Std', 'MFCC_9_Std', 'MFCC_10_Std', 'MFCC_11_Std', 'MFCC_12_Std', 'MFCC_13_Std', 'MFCC_1_Min', 'MFCC_2_Min', 'MFCC_3_Min', 'MFCC_4_Min', 'MFCC_5_Min', 'MFCC_6_Min', 'MFCC_7_Min', 'MFCC_8_Min', 'MFCC_9_Min', 'MFCC_10_Min', 'MFCC_11_Min', 'MFCC_12_Min', 'MFCC_13_Min', 'MFCC_1_Max', 'MFCC_2_Max', 'MFCC_3_Max', 'MFCC_4_Max', 'MFCC_5_

# MAMDANI FUZZY INFERENCE SYSTEM (FIS)

In [8]:
# ============================================================
# STEP 5
# MAMDANI FUZZY INFERENCE SYSTEM (FIS)
# ============================================================

import os
import numpy as np
import pandas as pd
import skfuzzy as fuzz

from skfuzzy import control as ctrl


# ------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------

BASE_PATH = r"F:\.0 Work\4358\Data"

STEP4_PATH = os.path.join(
    BASE_PATH,
    "step4_output"
)

TRAIN_PATH = os.path.join(
    STEP4_PATH,
    "train_dataset_scaled.csv"
)

TEST_PATH = os.path.join(
    STEP4_PATH,
    "test_dataset_scaled.csv"
)

OUTPUT_PATH = os.path.join(
    BASE_PATH,
    "step5_output"
)

os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)


print("=" * 70)
print("STEP 5: MAMDANI FUZZY INFERENCE SYSTEM")
print("=" * 70)


# ------------------------------------------------------------
# 2. LOAD STEP 4 DATA
# ------------------------------------------------------------

print("\n[1] Loading normalized training and testing data...")

train_df = pd.read_csv(
    TRAIN_PATH
)

test_df = pd.read_csv(
    TEST_PATH
)

print(
    "Training shape:",
    train_df.shape
)

print(
    "Testing shape:",
    test_df.shape
)


# ------------------------------------------------------------
# 3. CHECK REQUIRED FIS INPUTS
# ------------------------------------------------------------

fis_inputs = [
    "Pitch_Accuracy",
    "Rhythm_Accuracy",
    "Tempo_Stability"
]

target_column = (
    "Teaching_Effectiveness"
)

for feature in fis_inputs:

    if feature not in train_df.columns:

        raise ValueError(
            f"Missing FIS input: {feature}"
        )

print(
    "\nAll three FIS inputs are available."
)


# ------------------------------------------------------------
# 4. CREATE FUZZY VARIABLES
# ------------------------------------------------------------

print("\n[2] Creating fuzzy variables...")


# Universe of discourse
# All inputs have already been normalized to [0,1]

pitch = ctrl.Antecedent(
    np.linspace(0, 1, 101),
    "Pitch_Accuracy"
)

rhythm = ctrl.Antecedent(
    np.linspace(0, 1, 101),
    "Rhythm_Accuracy"
)

tempo = ctrl.Antecedent(
    np.linspace(0, 1, 101),
    "Tempo_Stability"
)


# Fuzzy output
learner_performance = ctrl.Consequent(
    np.linspace(0, 1, 101),
    "Learner_Performance"
)


# ------------------------------------------------------------
# 5. DEFINE MEMBERSHIP FUNCTIONS
# ------------------------------------------------------------

# Manuscript:
#
# Low    = trapezoidal [0, 0, 0.3, 0.5]
# Medium = triangular  [0.3, 0.5, 0.7]
# High   = trapezoidal [0.5, 0.7, 1, 1]
#
# Same linguistic structure is applied to all three inputs
# and the output.


def define_membership_functions(variable):

    variable["Low"] = fuzz.trapmf(
        variable.universe,
        [0.0, 0.0, 0.3, 0.5]
    )

    variable["Medium"] = fuzz.trimf(
        variable.universe,
        [0.3, 0.5, 0.7]
    )

    variable["High"] = fuzz.trapmf(
        variable.universe,
        [0.5, 0.7, 1.0, 1.0]
    )


define_membership_functions(
    pitch
)

define_membership_functions(
    rhythm
)

define_membership_functions(
    tempo
)

define_membership_functions(
    learner_performance
)


print(
    "Membership functions defined successfully."
)


# ------------------------------------------------------------
# 6. CREATE THE 27 FUZZY RULES
# ------------------------------------------------------------

print("\n[3] Creating 27 fuzzy rules...")


# Linguistic levels

levels = [
    "Low",
    "Medium",
    "High"
]


# ------------------------------------------------------------
# RULE DESIGN
# ------------------------------------------------------------
#
# The three inputs each have 3 linguistic levels:
#
# Pitch   : Low / Medium / High
# Rhythm  : Low / Medium / High
# Tempo   : Low / Medium / High
#
# Total possible combinations:
#
# 3 × 3 × 3 = 27
#
# The output is assigned according to the overall
# performance pattern:
#
# - Predominantly Low  -> Low
# - Predominantly High -> High
# - Mixed combinations -> Medium
#
# This produces the required 27-rule structure.


rules = []

rule_number = 1


for pitch_level in levels:

    for rhythm_level in levels:

        for tempo_level in levels:

            # Convert linguistic terms to numerical levels

            values = {
                "Low": 0,
                "Medium": 1,
                "High": 2
            }

            p = values[pitch_level]
            r = values[rhythm_level]
            t = values[tempo_level]

            average_level = (
                p + r + t
            ) / 3.0


            # Output decision

            if average_level < 0.67:

                output_level = "Low"

            elif average_level > 1.33:

                output_level = "High"

            else:

                output_level = "Medium"


            # Create fuzzy rule

            rule = ctrl.Rule(
                pitch[pitch_level]
                & rhythm[rhythm_level]
                & tempo[tempo_level],
                learner_performance[
                    output_level
                ]
            )

            rules.append(rule)

            print(
                f"Rule {rule_number:02d}: "
                f"Pitch={pitch_level}, "
                f"Rhythm={rhythm_level}, "
                f"Tempo={tempo_level} "
                f"--> Performance={output_level}"
            )

            rule_number += 1


print(
    "\nTotal fuzzy rules:",
    len(rules)
)


# ------------------------------------------------------------
# 7. CREATE FUZZY CONTROL SYSTEM
# ------------------------------------------------------------

print("\n[4] Building Mamdani fuzzy control system...")


fis_system = ctrl.ControlSystem(
    rules
)

fis_simulation = ctrl.ControlSystemSimulation(
    fis_system
)


print(
    "Mamdani FIS created successfully."
)


# ------------------------------------------------------------
# 8. FUNCTION FOR SINGLE LEARNER
# ------------------------------------------------------------

def calculate_fuzzy_performance(
    pitch_value,
    rhythm_value,
    tempo_value
):

    # Safety clipping to [0,1]

    pitch_value = np.clip(
        pitch_value,
        0.0,
        1.0
    )

    rhythm_value = np.clip(
        rhythm_value,
        0.0,
        1.0
    )

    tempo_value = np.clip(
        tempo_value,
        0.0,
        1.0
    )


    simulation = ctrl.ControlSystemSimulation(
        fis_system
    )

    simulation.input[
        "Pitch_Accuracy"
    ] = pitch_value

    simulation.input[
        "Rhythm_Accuracy"
    ] = rhythm_value

    simulation.input[
        "Tempo_Stability"
    ] = tempo_value


    simulation.compute()


    return simulation.output[
        "Learner_Performance"
    ]


# ------------------------------------------------------------
# 9. TEST FIS WITH ONE SAMPLE
# ------------------------------------------------------------

print("\n[5] Testing FIS with one learner...")

sample = train_df.iloc[0]

sample_pitch = sample[
    "Pitch_Accuracy"
]

sample_rhythm = sample[
    "Rhythm_Accuracy"
]

sample_tempo = sample[
    "Tempo_Stability"
]


sample_output = calculate_fuzzy_performance(
    sample_pitch,
    sample_rhythm,
    sample_tempo
)


print(
    "\nExample learner:"
)

print(
    "Pitch Accuracy :",
    round(sample_pitch, 4)
)

print(
    "Rhythm Accuracy:",
    round(sample_rhythm, 4)
)

print(
    "Tempo Stability:",
    round(sample_tempo, 4)
)

print(
    "Fuzzy Performance:",
    round(sample_output, 4)
)


# ------------------------------------------------------------
# 10. APPLY FIS TO TRAINING DATA
# ------------------------------------------------------------

print(
    "\n[6] Calculating fuzzy performance for training data..."
)


train_fuzzy_values = []

for _, row in train_df.iterrows():

    value = calculate_fuzzy_performance(
        row["Pitch_Accuracy"],
        row["Rhythm_Accuracy"],
        row["Tempo_Stability"]
    )

    train_fuzzy_values.append(
        value
    )


train_df[
    "Fuzzy_Learner_Performance"
] = train_fuzzy_values


# ------------------------------------------------------------
# 11. APPLY FIS TO TEST DATA
# ------------------------------------------------------------

print(
    "[7] Calculating fuzzy performance for testing data..."
)


test_fuzzy_values = []

for _, row in test_df.iterrows():

    value = calculate_fuzzy_performance(
        row["Pitch_Accuracy"],
        row["Rhythm_Accuracy"],
        row["Tempo_Stability"]
    )

    test_fuzzy_values.append(
        value
    )


test_df[
    "Fuzzy_Learner_Performance"
] = test_fuzzy_values


# ------------------------------------------------------------
# 12. FUZZY PERFORMANCE STATISTICS
# ------------------------------------------------------------

print(
    "\n[8] Fuzzy performance statistics..."
)

print(
    "\nTraining statistics:"
)

print(
    train_df[
        "Fuzzy_Learner_Performance"
    ].describe()
)


print(
    "\nTesting statistics:"
)

print(
    test_df[
        "Fuzzy_Learner_Performance"
    ].describe()
)


# ------------------------------------------------------------
# 13. CONVERT FUZZY SCORE INTO LINGUISTIC CATEGORY
# ------------------------------------------------------------

def fuzzy_category(score):

    if score < 0.33:

        return "Low"

    elif score < 0.67:

        return "Medium"

    else:

        return "High"


train_df[
    "Fuzzy_Performance_Level"
] = (
    train_df[
        "Fuzzy_Learner_Performance"
    ]
    .apply(fuzzy_category)
)


test_df[
    "Fuzzy_Performance_Level"
] = (
    test_df[
        "Fuzzy_Learner_Performance"
    ]
    .apply(fuzzy_category)
)


# ------------------------------------------------------------
# 14. DISPLAY CATEGORY DISTRIBUTION
# ------------------------------------------------------------

print(
    "\nTraining fuzzy performance levels:"
)

print(
    train_df[
        "Fuzzy_Performance_Level"
    ].value_counts()
)


print(
    "\nTesting fuzzy performance levels:"
)

print(
    test_df[
        "Fuzzy_Performance_Level"
    ].value_counts()
)


# ------------------------------------------------------------
# 15. SAVE TRAINING FIS DATA
# ------------------------------------------------------------

TRAIN_FIS_PATH = os.path.join(
    OUTPUT_PATH,
    "train_fuzzy_performance.csv"
)

train_df.to_csv(
    TRAIN_FIS_PATH,
    index=False
)


# ------------------------------------------------------------
# 16. SAVE TESTING FIS DATA
# ------------------------------------------------------------

TEST_FIS_PATH = os.path.join(
    OUTPUT_PATH,
    "test_fuzzy_performance.csv"
)

test_df.to_csv(
    TEST_FIS_PATH,
    index=False
)


# ------------------------------------------------------------
# 17. SAVE FUZZY RULE TABLE
# ------------------------------------------------------------

rule_records = []

rule_number = 1

for pitch_level in levels:

    for rhythm_level in levels:

        for tempo_level in levels:

            values = {
                "Low": 0,
                "Medium": 1,
                "High": 2
            }

            average_level = (
                values[pitch_level]
                + values[rhythm_level]
                + values[tempo_level]
            ) / 3.0


            if average_level < 0.67:

                output_level = "Low"

            elif average_level > 1.33:

                output_level = "High"

            else:

                output_level = "Medium"


            rule_records.append({
                "Rule": rule_number,
                "Pitch_Accuracy": pitch_level,
                "Rhythm_Accuracy": rhythm_level,
                "Tempo_Stability": tempo_level,
                "Learner_Performance": output_level
            })

            rule_number += 1


rule_df = pd.DataFrame(
    rule_records
)


RULE_PATH = os.path.join(
    OUTPUT_PATH,
    "fis_27_rules.csv"
)

rule_df.to_csv(
    RULE_PATH,
    index=False
)


# ------------------------------------------------------------
# 18. SAVE MEMBERSHIP FUNCTION PARAMETERS
# ------------------------------------------------------------

membership_info = pd.DataFrame({

    "Linguistic_Level": [
        "Low",
        "Medium",
        "High"
    ],

    "Membership_Type": [
        "Trapezoidal",
        "Triangular",
        "Trapezoidal"
    ],

    "Parameters": [
        "[0.0, 0.0, 0.3, 0.5]",
        "[0.3, 0.5, 0.7]",
        "[0.5, 0.7, 1.0, 1.0]"
    ]
})


MEMBERSHIP_PATH = os.path.join(
    OUTPUT_PATH,
    "fis_membership_functions.csv"
)

membership_info.to_csv(
    MEMBERSHIP_PATH,
    index=False
)


# ------------------------------------------------------------
# 19. FINAL OUTPUT
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("STEP 5 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    "\nTraining records:",
    len(train_df)
)

print(
    "Testing records:",
    len(test_df)
)

print(
    "Number of fuzzy rules:",
    len(rules)
)

print(
    "\nFuzzy output column:"
)

print(
    "Fuzzy_Learner_Performance"
)

print(
    "\nOutput files:"
)

print(
    TRAIN_FIS_PATH
)

print(
    TEST_FIS_PATH
)

print(
    RULE_PATH
)

print(
    MEMBERSHIP_PATH
)

STEP 5: MAMDANI FUZZY INFERENCE SYSTEM

[1] Loading normalized training and testing data...
Training shape: (799, 19)
Testing shape: (200, 19)

All three FIS inputs are available.

[2] Creating fuzzy variables...
Membership functions defined successfully.

[3] Creating 27 fuzzy rules...
Rule 01: Pitch=Low, Rhythm=Low, Tempo=Low --> Performance=Low
Rule 02: Pitch=Low, Rhythm=Low, Tempo=Medium --> Performance=Low
Rule 03: Pitch=Low, Rhythm=Low, Tempo=High --> Performance=Low
Rule 04: Pitch=Low, Rhythm=Medium, Tempo=Low --> Performance=Low
Rule 05: Pitch=Low, Rhythm=Medium, Tempo=Medium --> Performance=Low
Rule 06: Pitch=Low, Rhythm=Medium, Tempo=High --> Performance=Medium
Rule 07: Pitch=Low, Rhythm=High, Tempo=Low --> Performance=Low
Rule 08: Pitch=Low, Rhythm=High, Tempo=Medium --> Performance=Medium
Rule 09: Pitch=Low, Rhythm=High, Tempo=High --> Performance=High
Rule 10: Pitch=Medium, Rhythm=Low, Tempo=Low --> Performance=Low
Rule 11: Pitch=Medium, Rhythm=Low, Tempo=Medium --> Perfor

# SEQUENTIAL DATA PREPARATION FOR DTL-IntLSTM

In [9]:
# ============================================================
# STEP 6
# SEQUENTIAL DATA PREPARATION FOR DTL-IntLSTM
# ============================================================

import os
import random
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


# ------------------------------------------------------------
# 1. REPRODUCIBILITY
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


# ------------------------------------------------------------
# 2. PATH CONFIGURATION
# ------------------------------------------------------------

BASE_PATH = r"F:\.0 Work\4358\Data"

STEP5_PATH = os.path.join(
    BASE_PATH,
    "step5_output"
)

TRAIN_PATH = os.path.join(
    STEP5_PATH,
    "train_fuzzy_performance.csv"
)

TEST_PATH = os.path.join(
    STEP5_PATH,
    "test_fuzzy_performance.csv"
)

OUTPUT_PATH = os.path.join(
    BASE_PATH,
    "step6_output"
)

os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)


print("=" * 75)
print("STEP 6: SEQUENTIAL DATA PREPARATION FOR DTL-IntLSTM")
print("=" * 75)


# ------------------------------------------------------------
# 3. LOAD STEP 5 DATA
# ------------------------------------------------------------

print("\n[1] Loading Step 5 fuzzy-performance data...")

train_df = pd.read_csv(
    TRAIN_PATH
)

test_df = pd.read_csv(
    TEST_PATH
)

print(
    "Training shape:",
    train_df.shape
)

print(
    "Testing shape:",
    test_df.shape
)


# ------------------------------------------------------------
# 4. CHECK FUZZY OUTPUT
# ------------------------------------------------------------

required_fuzzy_column = (
    "Fuzzy_Learner_Performance"
)

if required_fuzzy_column not in train_df.columns:

    raise ValueError(
        f"""
Required column not found:

{required_fuzzy_column}

Please run Step 5 first.
"""
    )


# ------------------------------------------------------------
# 5. IDENTIFY INPUT FEATURES
# ------------------------------------------------------------

# These are the learner features prepared in Step 4/5.

base_features = [
    "Practice_Hours",
    "Assignment_Score",
    "Attendance_Rate",
    "Pitch_Accuracy",
    "Rhythm_Accuracy",
    "Tempo_Stability",
    "Feedback_Sentiment",
    "Feedback_Length"
]


# Existing dataset MFCC features

dataset_mfcc_features = [
    col
    for col in train_df.columns
    if col.startswith("MFCC_")
    and not col.startswith("Audio_")
]


# Fuzzy learner performance

fuzzy_features = [
    "Fuzzy_Learner_Performance"
]


# ------------------------------------------------------------
# 6. COMPLETE SEQUENTIAL FEATURE SET
# ------------------------------------------------------------

sequence_features = (
    base_features
    + dataset_mfcc_features
    + fuzzy_features
)


# Remove accidental duplicates

sequence_features = list(
    dict.fromkeys(
        sequence_features
    )
)


# Check availability

missing_sequence_features = [
    col
    for col in sequence_features
    if col not in train_df.columns
]

if missing_sequence_features:

    raise ValueError(
        f"""
Missing sequential features:

{missing_sequence_features}
"""
    )


print(
    "\nNumber of sequential input features:",
    len(sequence_features)
)

print(
    "\nSequential features:"
)

for i, feature in enumerate(
    sequence_features,
    start=1
):

    print(
        f"{i:02d}. {feature}"
    )


# ------------------------------------------------------------
# 7. PRESERVE ORIGINAL ROW ORDER
# ------------------------------------------------------------
#
# The manuscript specifies sequence length = 50, but does not
# define an explicit timestamp column or per-student sequence
# construction procedure.
#
# Therefore, this implementation preserves the dataset row
# order and constructs rolling windows.
# ------------------------------------------------------------

train_df = train_df.reset_index(
    drop=True
)

test_df = test_df.reset_index(
    drop=True
)


# ------------------------------------------------------------
# 8. CREATE NUMERIC MATRICES
# ------------------------------------------------------------

X_train = (
    train_df[
        sequence_features
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .values
)

X_test = (
    test_df[
        sequence_features
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .values
)


# ------------------------------------------------------------
# 9. CHECK NAN / INFINITE VALUES
# ------------------------------------------------------------

if not np.isfinite(
    X_train
).all():

    raise ValueError(
        "Training sequence features contain NaN or infinite values."
    )


if not np.isfinite(
    X_test
).all():

    raise ValueError(
        "Testing sequence features contain NaN or infinite values."
    )


print(
    "\nTraining feature matrix:",
    X_train.shape
)

print(
    "Testing feature matrix:",
    X_test.shape
)


# ------------------------------------------------------------
# 10. ENCODE TARGET LABELS
# ------------------------------------------------------------

print("\n[2] Encoding Teaching Effectiveness labels...")

label_encoder = LabelEncoder()

# Fit encoder on all known target classes

all_labels = pd.concat(
    [
        train_df[
            "Teaching_Effectiveness"
        ],
        test_df[
            "Teaching_Effectiveness"
        ]
    ]
).astype(str)

label_encoder.fit(
    all_labels
)


y_train = label_encoder.transform(
    train_df[
        "Teaching_Effectiveness"
    ].astype(str)
)

y_test = label_encoder.transform(
    test_df[
        "Teaching_Effectiveness"
    ].astype(str)
)


print(
    "\nLabel mapping:"
)

for encoded, label in enumerate(
    label_encoder.classes_
):

    print(
        f"{encoded} -> {label}"
    )


# ------------------------------------------------------------
# 11. SEQUENCE LENGTH
# ------------------------------------------------------------

SEQ_LEN = 50

print(
    "\nSequence length:",
    SEQ_LEN
)


# ------------------------------------------------------------
# 12. CREATE ROLLING SEQUENCES
# ------------------------------------------------------------

def create_sequences(
    X,
    y,
    sequence_length
):

    X_sequences = []
    y_sequences = []

    # A sequence of 50 observations predicts the label
    # associated with the final observation.

    for i in range(
        len(X) - sequence_length + 1
    ):

        start = i

        end = (
            i
            + sequence_length
        )

        sequence = X[
            start:end
        ]

        target = y[
            end - 1
        ]

        X_sequences.append(
            sequence
        )

        y_sequences.append(
            target
        )

    return (
        np.asarray(
            X_sequences,
            dtype=np.float32
        ),
        np.asarray(
            y_sequences,
            dtype=np.int64
        )
    )


X_train_seq, y_train_seq = (
    create_sequences(
        X_train,
        y_train,
        SEQ_LEN
    )
)


X_test_seq, y_test_seq = (
    create_sequences(
        X_test,
        y_test,
        SEQ_LEN
    )
)


# ------------------------------------------------------------
# 13. CHECK SEQUENCE SHAPES
# ------------------------------------------------------------

print("\n[3] Sequential data created.")

print(
    "\nTraining sequence shape:"
)

print(
    X_train_seq.shape
)

print(
    "Testing sequence shape:"
)

print(
    X_test_seq.shape
)


print(
    "\nExpected format:"
)

print(
    "(Number_of_sequences, 50, Number_of_features)"
)


# ------------------------------------------------------------
# 14. CHECK TARGET SHAPES
# ------------------------------------------------------------

print(
    "\nTraining target shape:",
    y_train_seq.shape
)

print(
    "Testing target shape:",
    y_test_seq.shape
)


# ------------------------------------------------------------
# 15. SEQUENCE TARGET DISTRIBUTION
# ------------------------------------------------------------

print(
    "\nTraining sequence target distribution:"
)

unique_train, counts_train = np.unique(
    y_train_seq,
    return_counts=True
)

for label_id, count in zip(
    unique_train,
    counts_train
):

    label_name = (
        label_encoder.inverse_transform(
            [label_id]
        )[0]
    )

    print(
        f"{label_id} ({label_name}): {count}"
    )


print(
    "\nTesting sequence target distribution:"
)

unique_test, counts_test = np.unique(
    y_test_seq,
    return_counts=True
)

for label_id, count in zip(
    unique_test,
    counts_test
):

    label_name = (
        label_encoder.inverse_transform(
            [label_id]
        )[0]
    )

    print(
        f"{label_id} ({label_name}): {count}"
    )


# ------------------------------------------------------------
# 16. CHECK SEQUENCE VALUES
# ------------------------------------------------------------

print(
    "\n[4] Sequence value verification..."
)

print(
    "Minimum training value:",
    X_train_seq.min()
)

print(
    "Maximum training value:",
    X_train_seq.max()
)

print(
    "Minimum testing value:",
    X_test_seq.min()
)

print(
    "Maximum testing value:",
    X_test_seq.max()
)


# ------------------------------------------------------------
# 17. SAVE NUMPY SEQUENCES
# ------------------------------------------------------------

TRAIN_X_PATH = os.path.join(
    OUTPUT_PATH,
    "X_train_sequences.npy"
)

TRAIN_Y_PATH = os.path.join(
    OUTPUT_PATH,
    "y_train_sequences.npy"
)

TEST_X_PATH = os.path.join(
    OUTPUT_PATH,
    "X_test_sequences.npy"
)

TEST_Y_PATH = os.path.join(
    OUTPUT_PATH,
    "y_test_sequences.npy"
)


np.save(
    TRAIN_X_PATH,
    X_train_seq
)

np.save(
    TRAIN_Y_PATH,
    y_train_seq
)

np.save(
    TEST_X_PATH,
    X_test_seq
)

np.save(
    TEST_Y_PATH,
    y_test_seq
)


# ------------------------------------------------------------
# 18. SAVE LABEL ENCODER
# ------------------------------------------------------------

LABEL_ENCODER_PATH = os.path.join(
    OUTPUT_PATH,
    "label_encoder.npy"
)

np.save(
    LABEL_ENCODER_PATH,
    label_encoder.classes_
)


# ------------------------------------------------------------
# 19. SAVE FEATURE LIST
# ------------------------------------------------------------

FEATURE_PATH = os.path.join(
    OUTPUT_PATH,
    "sequence_feature_list.csv"
)

pd.DataFrame({
    "Feature_Index": range(
        len(sequence_features)
    ),
    "Feature": sequence_features
}).to_csv(
    FEATURE_PATH,
    index=False
)


# ------------------------------------------------------------
# 20. SAVE SEQUENCE INFORMATION
# ------------------------------------------------------------

sequence_info = pd.DataFrame({

    "Parameter": [
        "Random_Seed",
        "Sequence_Length",
        "Training_Original_Records",
        "Testing_Original_Records",
        "Training_Sequences",
        "Testing_Sequences",
        "Number_of_Input_Features",
        "Number_of_Classes"
    ],

    "Value": [
        SEED,
        SEQ_LEN,
        len(train_df),
        len(test_df),
        len(X_train_seq),
        len(X_test_seq),
        len(sequence_features),
        len(label_encoder.classes_)
    ]
})


INFO_PATH = os.path.join(
    OUTPUT_PATH,
    "sequence_configuration.csv"
)

sequence_info.to_csv(
    INFO_PATH,
    index=False
)


# ------------------------------------------------------------
# 21. CREATE SMALL INSPECTION TABLE
# ------------------------------------------------------------

inspection_records = []

number_to_show = min(
    10,
    len(X_train_seq)
)

for i in range(
    number_to_show
):

    target_id = y_train_seq[i]

    target_name = (
        label_encoder.inverse_transform(
            [target_id]
        )[0]
    )

    inspection_records.append({

        "Sequence_ID": i,

        "Start_Record": i,

        "End_Record": (
            i + SEQ_LEN - 1
        ),

        "Sequence_Length": SEQ_LEN,

        "Target_ID": target_id,

        "Target_Label": target_name
    })


inspection_df = pd.DataFrame(
    inspection_records
)


INSPECTION_PATH = os.path.join(
    OUTPUT_PATH,
    "sequence_inspection.csv"
)

inspection_df.to_csv(
    INSPECTION_PATH,
    index=False
)


# ------------------------------------------------------------
# 22. FINAL SUMMARY
# ------------------------------------------------------------

print("\n")
print("=" * 75)
print("STEP 6 COMPLETED SUCCESSFULLY")
print("=" * 75)

print(
    "\nOriginal training records:",
    len(train_df)
)

print(
    "Original testing records:",
    len(test_df)
)

print(
    "\nSequence length:",
    SEQ_LEN
)

print(
    "Number of input features:",
    len(sequence_features)
)

print(
    "\nTraining sequences:",
    len(X_train_seq)
)

print(
    "Testing sequences:",
    len(X_test_seq)
)

print(
    "\nTraining X shape:",
    X_train_seq.shape
)

print(
    "Training y shape:",
    y_train_seq.shape
)

print(
    "\nTesting X shape:",
    X_test_seq.shape
)

print(
    "Testing y shape:",
    y_test_seq.shape
)

print(
    "\nSaved files:"
)

print(
    "1.",
    TRAIN_X_PATH
)

print(
    "2.",
    TRAIN_Y_PATH
)

print(
    "3.",
    TEST_X_PATH
)

print(
    "4.",
    TEST_Y_PATH
)

print(
    "5.",
    LABEL_ENCODER_PATH
)

print(
    "6.",
    FEATURE_PATH
)

print(
    "7.",
    INFO_PATH
)

print(
    "8.",
    INSPECTION_PATH
)

STEP 6: SEQUENTIAL DATA PREPARATION FOR DTL-IntLSTM

[1] Loading Step 5 fuzzy-performance data...
Training shape: (799, 21)
Testing shape: (200, 21)

Number of sequential input features: 19

Sequential features:
01. Practice_Hours
02. Assignment_Score
03. Attendance_Rate
04. Pitch_Accuracy
05. Rhythm_Accuracy
06. Tempo_Stability
07. Feedback_Sentiment
08. Feedback_Length
09. MFCC_1
10. MFCC_2
11. MFCC_3
12. MFCC_4
13. MFCC_5
14. MFCC_6
15. MFCC_7
16. MFCC_8
17. MFCC_9
18. MFCC_10
19. Fuzzy_Learner_Performance

Training feature matrix: (799, 19)
Testing feature matrix: (200, 19)

[2] Encoding Teaching Effectiveness labels...

Label mapping:
0 -> High
1 -> Low
2 -> Medium

Sequence length: 50

[3] Sequential data created.

Training sequence shape:
(750, 50, 19)
Testing sequence shape:
(151, 50, 19)

Expected format:
(Number_of_sequences, 50, Number_of_features)

Training target shape: (750,)
Testing target shape: (151,)

Training sequence target distribution:
0 (High): 271
1 (Low): 238
2

# DTL-IntLSTM MODEL CONSTRUCTION AND TRAINING

In [12]:
# ============================================================
# STEP 7
# DTL-IntLSTM MODEL CONSTRUCTION AND TRAINING
# ============================================================

import os
import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# ------------------------------------------------------------
# 1. REPRODUCIBILITY
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# More deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ------------------------------------------------------------
# 2. PATH CONFIGURATION
# ------------------------------------------------------------

BASE_PATH = r"F:\.0 Work\4358\Data"

STEP6_PATH = os.path.join(
    BASE_PATH,
    "step6_output"
)

OUTPUT_PATH = os.path.join(
    BASE_PATH,
    "step7_output"
)

os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)


# ------------------------------------------------------------
# 3. DEVICE
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 75)
print("STEP 7: DTL-IntLSTM MODEL CONSTRUCTION AND TRAINING")
print("=" * 75)

print(
    "\nComputational device:",
    DEVICE
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ------------------------------------------------------------
# 4. LOAD STEP 6 SEQUENCES
# ------------------------------------------------------------

print("\n[1] Loading sequential data...")

X_train = np.load(
    os.path.join(
        STEP6_PATH,
        "X_train_sequences.npy"
    )
)

y_train = np.load(
    os.path.join(
        STEP6_PATH,
        "y_train_sequences.npy"
    )
)

X_test = np.load(
    os.path.join(
        STEP6_PATH,
        "X_test_sequences.npy"
    )
)

y_test = np.load(
    os.path.join(
        STEP6_PATH,
        "y_test_sequences.npy"
    )
)


print(
    "\nX_train:",
    X_train.shape
)

print(
    "y_train:",
    y_train.shape
)

print(
    "X_test:",
    X_test.shape
)

print(
    "y_test:",
    y_test.shape
)


# ------------------------------------------------------------
# 5. DETERMINE MODEL DIMENSIONS
# ------------------------------------------------------------

SEQ_LEN = X_train.shape[1]

INPUT_SIZE = X_train.shape[2]

NUM_CLASSES = len(
    np.unique(
        np.concatenate(
            [
                y_train,
                y_test
            ]
        )
    )
)

HIDDEN_SIZE = 128

NUM_LAYERS = 2

DROPOUT = 0.30

LEARNING_RATE = 0.001

BATCH_SIZE = 32

MAX_EPOCHS = 50

PATIENCE = 5


print("\nModel configuration:")

print(
    "Sequence length:",
    SEQ_LEN
)

print(
    "Input features:",
    INPUT_SIZE
)

print(
    "Hidden units:",
    HIDDEN_SIZE
)

print(
    "LSTM layers:",
    NUM_LAYERS
)

print(
    "Dropout:",
    DROPOUT
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Maximum epochs:",
    MAX_EPOCHS
)

print(
    "Early stopping patience:",
    PATIENCE
)

print(
    "Number of classes:",
    NUM_CLASSES
)


# ------------------------------------------------------------
# 6. CONVERT NUMPY → PYTORCH TENSORS
# ------------------------------------------------------------

print("\n[2] Converting data to PyTorch tensors...")

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)


# ------------------------------------------------------------
# 7. CREATE DATASETS
# ------------------------------------------------------------

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)


# ------------------------------------------------------------
# 8. CREATE DATALOADERS
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


print(
    "\nTraining batches:",
    len(train_loader)
)

print(
    "Testing batches:",
    len(test_loader)
)


# ============================================================
# 9. DEFINE DTL-INTLSTM MODEL
# ============================================================

class DTLIntLSTM(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        num_classes,
        dropout
    ):

        super(DTLIntLSTM, self).__init__()


        # ----------------------------------------------------
        # Two-layer LSTM
        # ----------------------------------------------------

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )


        # ----------------------------------------------------
        # Dropout before classification
        # ----------------------------------------------------

        self.dropout = nn.Dropout(
            p=dropout
        )


        # ----------------------------------------------------
        # Fully connected classification layer
        # ----------------------------------------------------

        self.fc = nn.Linear(
            hidden_size,
            num_classes
        )


    def forward(self, x):

        # x:
        # [batch, sequence_length, features]

        lstm_output, (hidden, cell) = (
            self.lstm(x)
        )


        # Final hidden state of the second LSTM layer

        final_hidden = hidden[-1]


        # Dropout

        final_hidden = self.dropout(
            final_hidden
        )


        # Classification

        output = self.fc(
            final_hidden
        )


        return output


# ------------------------------------------------------------
# 10. CREATE MODEL
# ------------------------------------------------------------

print("\n[3] Creating DTL-IntLSTM model...")

model = DTLIntLSTM(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
)


model = model.to(
    DEVICE
)


print("\nModel architecture:")
print(model)


# ------------------------------------------------------------
# 11. COUNT TRAINABLE PARAMETERS
# ------------------------------------------------------------

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "\nTrainable parameters:",
    trainable_parameters
)


# ------------------------------------------------------------
# 12. LOSS FUNCTION
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# 13. OPTIMIZER
# ------------------------------------------------------------

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


# ------------------------------------------------------------
# 14. VALIDATION SPLIT
# ------------------------------------------------------------
#
# The manuscript reports validation-based early stopping.
#
# We create a validation subset from the training sequences.
# The test set remains untouched until final evaluation.
# ------------------------------------------------------------

print(
    "\n[4] Creating validation split..."
)


validation_fraction = 0.15

train_indices = np.arange(
    len(X_train)
)

train_indices_sub, val_indices = (
    train_test_split(
        train_indices,
        test_size=validation_fraction,
        random_state=SEED,
        stratify=y_train
    )
)


X_train_sub = X_train[
    train_indices_sub
]

y_train_sub = y_train[
    train_indices_sub
]

X_val = X_train[
    val_indices
]

y_val = y_train[
    val_indices
]


print(
    "Training sequences:",
    len(X_train_sub)
)

print(
    "Validation sequences:",
    len(X_val)
)


# ------------------------------------------------------------
# 15. CREATE TRAINING/VALIDATION DATASETS
# ------------------------------------------------------------

train_sub_dataset = TensorDataset(
    torch.tensor(
        X_train_sub,
        dtype=torch.float32
    ),
    torch.tensor(
        y_train_sub,
        dtype=torch.long
    )
)


val_dataset = TensorDataset(
    torch.tensor(
        X_val,
        dtype=torch.float32
    ),
    torch.tensor(
        y_val,
        dtype=torch.long
    )
)


train_sub_loader = DataLoader(
    train_sub_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


# ------------------------------------------------------------
# 16. ACCURACY FUNCTION
# ------------------------------------------------------------

def calculate_accuracy(
    predictions,
    targets
):

    predicted_labels = (
        torch.argmax(
            predictions,
            dim=1
        )
    )

    correct = (
        predicted_labels == targets
    ).sum().item()

    total = targets.size(0)

    return (
        correct / total
        if total > 0
        else 0.0
    )


# ------------------------------------------------------------
# 17. VALIDATION FUNCTION
# ------------------------------------------------------------

def evaluate_model(
    model,
    data_loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0.0

    total_correct = 0

    total_samples = 0


    with torch.no_grad():

        for inputs, targets in data_loader:

            inputs = inputs.to(
                device
            )

            targets = targets.to(
                device
            )


            outputs = model(
                inputs
            )


            loss = criterion(
                outputs,
                targets
            )


            batch_size = (
                targets.size(0)
            )

            total_loss += (
                loss.item()
                * batch_size
            )


            predictions = torch.argmax(
                outputs,
                dim=1
            )


            total_correct += (
                predictions == targets
            ).sum().item()


            total_samples += (
                batch_size
            )


    average_loss = (
        total_loss / total_samples
        if total_samples > 0
        else 0.0
    )

    accuracy = (
        total_correct / total_samples
        if total_samples > 0
        else 0.0
    )


    return (
        average_loss,
        accuracy
    )


# ============================================================
# 18. TRAINING LOOP
# ============================================================

print("\n[5] Starting DTL-IntLSTM training...")
print("-" * 75)


history = {

    "epoch": [],

    "train_loss": [],

    "train_accuracy": [],

    "val_loss": [],

    "val_accuracy": []
}


best_val_loss = float(
    "inf"
)

best_epoch = 0

patience_counter = 0

BEST_MODEL_PATH = os.path.join(
    OUTPUT_PATH,
    "best_dtl_intlstm_model.pth"
)


for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    # --------------------------------------------------------
    # TRAIN MODE
    # --------------------------------------------------------

    model.train()

    running_loss = 0.0

    running_correct = 0

    running_samples = 0


    for inputs, targets in train_sub_loader:

        inputs = inputs.to(
            DEVICE
        )

        targets = targets.to(
            DEVICE
        )


        # Clear gradients

        optimizer.zero_grad()


        # Forward pass

        outputs = model(
            inputs
        )


        # Loss

        loss = criterion(
            outputs,
            targets
        )


        # Backpropagation

        loss.backward()


        # Update parameters

        optimizer.step()


        # Statistics

        batch_size = (
            targets.size(0)
        )

        running_loss += (
            loss.item()
            * batch_size
        )


        predictions = torch.argmax(
            outputs,
            dim=1
        )


        running_correct += (
            predictions == targets
        ).sum().item()


        running_samples += (
            batch_size
        )


    train_loss = (
        running_loss /
        running_samples
    )

    train_accuracy = (
        running_correct /
        running_samples
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    val_loss, val_accuracy = (
        evaluate_model(
            model,
            val_loader,
            criterion,
            DEVICE
        )
    )


    # --------------------------------------------------------
    # STORE HISTORY
    # --------------------------------------------------------

    history["epoch"].append(
        epoch
    )

    history["train_loss"].append(
        train_loss
    )

    history["train_accuracy"].append(
        train_accuracy
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_accuracy"].append(
        val_accuracy
    )


    # --------------------------------------------------------
    # PRINT EPOCH
    # --------------------------------------------------------

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy * 100:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy * 100:.2f}%"
    )


    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_epoch = epoch

        patience_counter = 0


        # Save best model

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )


    else:

        patience_counter += 1


        if patience_counter >= PATIENCE:

            print(
                f"\nEarly stopping triggered at "
                f"epoch {epoch}."
            )

            break


# ------------------------------------------------------------
# 19. LOAD BEST MODEL
# ------------------------------------------------------------

print(
    "\n[6] Loading best validation model..."
)

model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE
    )
)

model.eval()


print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation loss:",
    best_val_loss
)


# ============================================================
# 20. FINAL TEST EVALUATION
# ============================================================

print(
    "\n[7] Evaluating on held-out test sequences..."
)


test_loss, test_accuracy = (
    evaluate_model(
        model,
        test_loader,
        criterion,
        DEVICE
    )
)


print(
    "\nTest Loss:",
    round(test_loss, 6)
)

print(
    "Test Accuracy:",
    round(
        test_accuracy * 100,
        2
    ),
    "%"
)


# ============================================================
# 21. SAVE TRAINING HISTORY
# ============================================================

history_df = pd.DataFrame(
    history
)

HISTORY_PATH = os.path.join(
    OUTPUT_PATH,
    "dtl_intlstm_training_history.csv"
)

history_df.to_csv(
    HISTORY_PATH,
    index=False
)


# ============================================================
# 22. SAVE MODEL CONFIGURATION
# ============================================================

model_configuration = {

    "seed": SEED,

    "sequence_length": SEQ_LEN,

    "input_features": INPUT_SIZE,

    "hidden_size": HIDDEN_SIZE,

    "num_layers": NUM_LAYERS,

    "dropout": DROPOUT,

    "learning_rate": LEARNING_RATE,

    "batch_size": BATCH_SIZE,

    "maximum_epochs": MAX_EPOCHS,

    "early_stopping_patience": PATIENCE,

    "num_classes": NUM_CLASSES,

    "best_epoch": best_epoch,

    "best_validation_loss": float(
        best_val_loss
    ),

    "test_loss": float(
        test_loss
    ),

    "test_accuracy": float(
        test_accuracy
    ),

    "device": str(DEVICE),

    "trainable_parameters": int(
        trainable_parameters
    )
}


CONFIG_PATH = os.path.join(
    OUTPUT_PATH,
    "dtl_intlstm_configuration.json"
)


with open(
    CONFIG_PATH,
    "w"
) as f:

    json.dump(
        model_configuration,
        f,
        indent=4
    )


# ============================================================
# 23. SAVE TEST PREDICTIONS
# ============================================================

print(
    "\n[8] Generating test predictions..."
)


model.eval()

all_predictions = []

all_probabilities = []

all_targets = []


with torch.no_grad():

    for inputs, targets in test_loader:

        inputs = inputs.to(
            DEVICE
        )

        outputs = model(
            inputs
        )


        probabilities = torch.softmax(
            outputs,
            dim=1
        )


        predictions = torch.argmax(
            probabilities,
            dim=1
        )


        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )

        all_targets.extend(
            targets.numpy()
        )


all_predictions = np.asarray(
    all_predictions
)

all_targets = np.asarray(
    all_targets
)

all_probabilities = np.asarray(
    all_probabilities
)


prediction_df = pd.DataFrame({

    "Actual_Class": all_targets,

    "Predicted_Class": all_predictions
})


for class_id in range(
    NUM_CLASSES
):

    prediction_df[
        f"Probability_Class_{class_id}"
    ] = all_probabilities[
        :, class_id
    ]


PREDICTION_PATH = os.path.join(
    OUTPUT_PATH,
    "dtl_intlstm_test_predictions.csv"
)

prediction_df.to_csv(
    PREDICTION_PATH,
    index=False
)



# ============================================================
# 26. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 75)
print("STEP 7 COMPLETED SUCCESSFULLY")
print("=" * 75)

print(
    "\nModel: DTL-IntLSTM"
)

print(
    "LSTM layers:",
    NUM_LAYERS
)

print(
    "Hidden units:",
    HIDDEN_SIZE
)

print(
    "Dropout:",
    DROPOUT
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "\nFinal test loss:",
    f"{test_loss:.4f}"
)

print(
    "Final test accuracy:",
    f"{test_accuracy * 100:.2f}%"
)

print(
    "\nTrainable parameters:",
    trainable_parameters
)

print("\nSaved outputs:")

print(
    "1.",
    BEST_MODEL_PATH
)

print(
    "2.",
    HISTORY_PATH
)

print(
    "3.",
    CONFIG_PATH
)

print(
    "4.",
    PREDICTION_PATH
)

print(
    "5.",
    ACCURACY_GRAPH_PATH
)

print(
    "6.",
    LOSS_GRAPH_PATH
)

STEP 7: DTL-IntLSTM MODEL CONSTRUCTION AND TRAINING

Computational device: cpu

[1] Loading sequential data...

X_train: (750, 50, 19)
y_train: (750,)
X_test: (151, 50, 19)
y_test: (151,)

Model configuration:
Sequence length: 50
Input features: 19
Hidden units: 128
LSTM layers: 2
Dropout: 0.3
Learning rate: 0.001
Batch size: 32
Maximum epochs: 50
Early stopping patience: 5
Number of classes: 3

[2] Converting data to PyTorch tensors...

Training batches: 24
Testing batches: 5

[3] Creating DTL-IntLSTM model...

Model architecture:
DTLIntLSTM(
  (lstm): LSTM(19, 128, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=3, bias=True)
)

Trainable parameters: 208771

[4] Creating validation split...
Training sequences: 637
Validation sequences: 113

[5] Starting DTL-IntLSTM training...
---------------------------------------------------------------------------
Epoch 01/50 | Train Loss: 1.1013 | Train Acc: 34.

# DTLBO OPTIMIZATION OF DTL-IntLSTM

In [ ]:
# ============================================================
# STEP 8
# DTLBO OPTIMIZATION OF DTL-IntLSTM
# ============================================================
#
# Objective:
# Optimize IntLSTM hyperparameters using validation
# cross-entropy loss.
#
# Test data is NOT used during optimization.
#
# Manuscript settings:
#   Population size = 30
#   Maximum iterations = 100
#   Fitness = validation cross-entropy
#
# ============================================================

import os
import json
import random
import copy
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader
)


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# 2. PATH CONFIGURATION
# ============================================================

BASE_PATH = r"F:\.0 Work\4358\Data"

STEP6_PATH = os.path.join(
    BASE_PATH,
    "step6_output"
)

STEP7_PATH = os.path.join(
    BASE_PATH,
    "step7_output"
)

OUTPUT_PATH = os.path.join(
    BASE_PATH,
    "step8_output"
)

os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)


# ============================================================
# 3. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 80)
print("STEP 8: DTLBO OPTIMIZATION OF DTL-IntLSTM")
print("=" * 80)

print(
    "\nDevice:",
    DEVICE
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# 4. LOAD STEP 6 DATA
# ============================================================

print("\n[1] Loading sequential data...")

X_train_all = np.load(
    os.path.join(
        STEP6_PATH,
        "X_train_sequences.npy"
    )
)

y_train_all = np.load(
    os.path.join(
        STEP6_PATH,
        "y_train_sequences.npy"
    )
)

X_test = np.load(
    os.path.join(
        STEP6_PATH,
        "X_test_sequences.npy"
    )
)

y_test = np.load(
    os.path.join(
        STEP6_PATH,
        "y_test_sequences.npy"
    )
)


print(
    "Training sequences:",
    X_train_all.shape
)

print(
    "Testing sequences:",
    X_test.shape
)


# ============================================================
# 5. RECREATE TRAIN / VALIDATION SPLIT
# ============================================================

print(
    "\n[2] Creating optimization training/validation split..."
)

from sklearn.model_selection import train_test_split


train_indices = np.arange(
    len(X_train_all)
)

train_indices_sub, val_indices = (
    train_test_split(
        train_indices,
        test_size=0.15,
        random_state=SEED,
        stratify=y_train_all
    )
)


X_train = X_train_all[
    train_indices_sub
]

y_train = y_train_all[
    train_indices_sub
]

X_val = X_train_all[
    val_indices
]

y_val = y_train_all[
    val_indices
]


print(
    "Optimization training sequences:",
    len(X_train)
)

print(
    "Validation sequences:",
    len(X_val)
)

print(
    "Held-out test sequences:",
    len(X_test)
)


# ============================================================
# 6. DETERMINE DIMENSIONS
# ============================================================

SEQ_LEN = X_train.shape[1]

INPUT_SIZE = X_train.shape[2]

NUM_CLASSES = len(
    np.unique(
        np.concatenate(
            [
                y_train_all,
                y_test
            ]
        )
    )
)


print(
    "\nSequence length:",
    SEQ_LEN
)

print(
    "Input features:",
    INPUT_SIZE
)

print(
    "Number of classes:",
    NUM_CLASSES
)


# ============================================================
# 7. CREATE PYTORCH DATASETS
# ============================================================

train_dataset = TensorDataset(

    torch.tensor(
        X_train,
        dtype=torch.float32
    ),

    torch.tensor(
        y_train,
        dtype=torch.long
    )
)


val_dataset = TensorDataset(

    torch.tensor(
        X_val,
        dtype=torch.float32
    ),

    torch.tensor(
        y_val,
        dtype=torch.long
    )
)


# ============================================================
# 8. DTL-INTLSTM MODEL
# ============================================================

class DTLIntLSTM(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        num_classes,
        dropout
    ):

        super().__init__()

        self.lstm = nn.LSTM(

            input_size=input_size,

            hidden_size=hidden_size,

            num_layers=num_layers,

            batch_first=True,

            dropout=(
                dropout
                if num_layers > 1
                else 0.0
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.fc = nn.Linear(
            hidden_size,
            num_classes
        )


    def forward(self, x):

        output, (
            hidden,
            cell
        ) = self.lstm(x)

        final_hidden = hidden[-1]

        final_hidden = self.dropout(
            final_hidden
        )

        logits = self.fc(
            final_hidden
        )

        return logits


# ============================================================
# 9. HYPERPARAMETER SEARCH SPACE
# ============================================================
#
# IMPORTANT:
# The manuscript gives the DTLBO population and iteration
# settings, but the supplied methodology does not provide
# complete numerical bounds for every optimized parameter.
#
# Therefore these bounds are explicit implementation choices.
#
# ============================================================

SEARCH_SPACE = {

    # Hidden units
    "hidden_size": {
        "type": "int",
        "min": 64,
        "max": 256,
        "step": 32
    },

    # Number of LSTM layers
    "num_layers": {
        "type": "int",
        "min": 1,
        "max": 3,
        "step": 1
    },

    # Dropout
    "dropout": {
        "type": "float",
        "min": 0.10,
        "max": 0.50
    },

    # Learning rate
    "learning_rate": {
        "type": "float",
        "min": 0.0001,
        "max": 0.005
    },

    # Batch size
    "batch_size": {
        "type": "categorical",
        "values": [
            16,
            32,
            64
        ]
    }
}


# ============================================================
# 10. POPULATION / ITERATION SETTINGS
# ============================================================

POPULATION_SIZE = 30

MAX_ITERATIONS = 10

EVALUATION_EPOCHS = 10

EARLY_STOPPING_PATIENCE = 3


print(
    "\nDTLBO configuration:"
)

print(
    "Population size:",
    POPULATION_SIZE
)

print(
    "Maximum iterations:",
    MAX_ITERATIONS
)

print(
    "Fitness:",
    "Validation Cross-Entropy Loss"
)

print(
    "Evaluation epochs:",
    EVALUATION_EPOCHS
)


# ============================================================
# 11. PARAMETER ENCODING
# ============================================================

PARAMETER_NAMES = [
    "hidden_size",
    "num_layers",
    "dropout",
    "learning_rate",
    "batch_size"
]


# Continuous representation:
#
# hidden_size    → [64, 256]
# num_layers     → [1, 3]
# dropout        → [0.10, 0.50]
# learning_rate  → [0.0001, 0.005]
# batch_size     → categorical encoded [0, 1, 2]


LOWER_BOUNDS = np.array([
    64,
    1,
    0.10,
    0.0001,
    0
], dtype=float)


UPPER_BOUNDS = np.array([
    256,
    3,
    0.50,
    0.005,
    2
], dtype=float)


# ============================================================
# 12. DECODE PARTICLE
# ============================================================

BATCH_VALUES = [
    16,
    32,
    64
]


def decode_particle(position):

    hidden_size = int(
        round(
            position[0] / 32
        ) * 32
    )

    hidden_size = int(
        np.clip(
            hidden_size,
            64,
            256
        )
    )


    num_layers = int(
        round(
            position[1]
        )
    )

    num_layers = int(
        np.clip(
            num_layers,
            1,
            3
        )
    )


    dropout = float(
        np.clip(
            position[2],
            0.10,
            0.50
        )
    )


    learning_rate = float(
        np.clip(
            position[3],
            0.0001,
            0.005
        )
    )


    batch_index = int(
        round(
            position[4]
        )
    )

    batch_index = int(
        np.clip(
            batch_index,
            0,
            len(BATCH_VALUES) - 1
        )
    )


    batch_size = BATCH_VALUES[
        batch_index
    ]


    return {

        "hidden_size":
            hidden_size,

        "num_layers":
            num_layers,

        "dropout":
            dropout,

        "learning_rate":
            learning_rate,

        "batch_size":
            batch_size
    }


# ============================================================
# 13. CREATE INITIAL POPULATION
# ============================================================

print(
    "\n[3] Initializing DTLBO population..."
)

population = np.random.uniform(

    LOWER_BOUNDS,

    UPPER_BOUNDS,

    size=(
        POPULATION_SIZE,
        len(PARAMETER_NAMES)
    )
)


# ============================================================
# 14. TRAIN ONE CANDIDATE
# ============================================================

def evaluate_candidate(
    position
):

    params = decode_particle(
        position
    )


    # --------------------------------------------------------
    # Create fresh model
    # --------------------------------------------------------

    model = DTLIntLSTM(

        input_size=INPUT_SIZE,

        hidden_size=params[
            "hidden_size"
        ],

        num_layers=params[
            "num_layers"
        ],

        num_classes=NUM_CLASSES,

        dropout=params[
            "dropout"
        ]
    ).to(DEVICE)


    criterion = nn.CrossEntropyLoss()


    optimizer = torch.optim.Adam(

        model.parameters(),

        lr=params[
            "learning_rate"
        ]
    )


    train_loader = DataLoader(

        train_dataset,

        batch_size=params[
            "batch_size"
        ],

        shuffle=True,

        num_workers=0
    )


    val_loader = DataLoader(

        val_dataset,

        batch_size=params[
            "batch_size"
        ],

        shuffle=False,

        num_workers=0
    )


    best_val_loss = float(
        "inf"
    )

    patience_counter = 0


    # --------------------------------------------------------
    # Candidate training
    # --------------------------------------------------------

    for epoch in range(
        EVALUATION_EPOCHS
    ):

        model.train()


        for inputs, targets in train_loader:

            inputs = inputs.to(
                DEVICE
            )

            targets = targets.to(
                DEVICE
            )


            optimizer.zero_grad()


            outputs = model(
                inputs
            )


            loss = criterion(
                outputs,
                targets
            )


            loss.backward()


            optimizer.step()


        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        model.eval()

        total_loss = 0.0

        total_samples = 0


        with torch.no_grad():

            for inputs, targets in val_loader:

                inputs = inputs.to(
                    DEVICE
                )

                targets = targets.to(
                    DEVICE
                )


                outputs = model(
                    inputs
                )


                loss = criterion(
                    outputs,
                    targets
                )


                batch_n = targets.size(0)

                total_loss += (
                    loss.item()
                    * batch_n
                )

                total_samples += batch_n


        val_loss = (
            total_loss /
            total_samples
        )


        if val_loss < best_val_loss:

            best_val_loss = val_loss

            patience_counter = 0

        else:

            patience_counter += 1

            if (
                patience_counter
                >= EARLY_STOPPING_PATIENCE
            ):

                break


    return (
        best_val_loss,
        params
    )


# ============================================================
# 15. INITIAL FITNESS EVALUATION
# ============================================================

print(
    "\n[4] Evaluating initial population..."
)

fitness = np.zeros(
    POPULATION_SIZE
)

decoded_population = []


optimization_start = time.time()


for i in range(
    POPULATION_SIZE
):

    loss, params = (
        evaluate_candidate(
            population[i]
        )
    )

    fitness[i] = loss

    decoded_population.append(
        params
    )


    print(
        f"Candidate {i + 1:02d}/"
        f"{POPULATION_SIZE} | "
        f"Validation Loss = "
        f"{loss:.6f} | "
        f"Params = {params}"
    )


# ============================================================
# 16. IDENTIFY BEST SOLUTION
# ============================================================

best_index = np.argmin(
    fitness
)

best_position = (
    population[
        best_index
    ].copy()
)

best_fitness = float(
    fitness[
        best_index
    ]
)

best_parameters = decode_particle(
    best_position
)


print(
    "\nInitial best validation loss:",
    best_fitness
)

print(
    "Initial best parameters:",
    best_parameters
)


# ============================================================
# 17. DYNAMIC TEACHING FACTOR
# ============================================================

def dynamic_teaching_factor(
    iteration,
    max_iterations
):

    # Dynamic factor decreases as optimization progresses.

    progress = (
        iteration /
        max_iterations
    )

    TF = (
        2.0
        - progress
    )

    return TF


# ============================================================
# 18. DTLBO OPTIMIZATION
# ============================================================

print(
    "\n[5] Starting DTLBO optimization..."
)

print("-" * 80)


convergence_history = []


for iteration in range(
    1,
    MAX_ITERATIONS + 1
):

    # ========================================================
    # TEACHER PHASE
    # ========================================================

    teacher_index = np.argmin(
        fitness
    )

    teacher = (
        population[
            teacher_index
        ].copy()
    )


    mean_solution = (
        np.mean(
            population,
            axis=0
        )
    )


    TF = dynamic_teaching_factor(
        iteration,
        MAX_ITERATIONS
    )


    for i in range(
        POPULATION_SIZE
    ):

        random_factor = np.random.rand()

        new_position = (
            population[i]
            + random_factor
            * (
                teacher
                - TF
                * mean_solution
            )
        )


        new_position = np.clip(
            new_position,
            LOWER_BOUNDS,
            UPPER_BOUNDS
        )


        new_fitness, _ = (
            evaluate_candidate(
                new_position
            )
        )


        if new_fitness < fitness[i]:

            population[i] = (
                new_position
            )

            fitness[i] = (
                new_fitness
            )


    # ========================================================
    # LEARNER PHASE
    # ========================================================

    for i in range(
        POPULATION_SIZE
    ):

        # Select another learner

        j = np.random.randint(
            0,
            POPULATION_SIZE
        )

        while j == i:

            j = np.random.randint(
                0,
                POPULATION_SIZE
            )


        learner_i = (
            population[i]
        )

        learner_j = (
            population[j]
        )


        r = np.random.rand()


        # Lower fitness means better solution.

        if fitness[i] < fitness[j]:

            new_position = (
                learner_i
                + r
                * (
                    learner_i
                    - learner_j
                )
            )

        else:

            new_position = (
                learner_i
                + r
                * (
                    learner_j
                    - learner_i
                )
            )


        new_position = np.clip(
            new_position,
            LOWER_BOUNDS,
            UPPER_BOUNDS
        )


        new_fitness, _ = (
            evaluate_candidate(
                new_position
            )
        )


        if new_fitness < fitness[i]:

            population[i] = (
                new_position
            )

            fitness[i] = (
                new_fitness
            )


    # ========================================================
    # GLOBAL BEST
    # ========================================================

    current_best_index = (
        np.argmin(
            fitness
        )
    )


    current_best_fitness = float(
        fitness[
            current_best_index
        ]
    )


    if (
        current_best_fitness
        < best_fitness
    ):

        best_fitness = (
            current_best_fitness
        )

        best_position = (
            population[
                current_best_index
            ].copy()
        )

        best_parameters = (
            decode_particle(
                best_position
            )
        )


    convergence_history.append({

        "Iteration":
            iteration,

        "Best_Validation_Loss":
            best_fitness,

        "Teaching_Factor":
            TF
    })


    print(
        f"Iteration "
        f"{iteration:03d}/"
        f"{MAX_ITERATIONS} | "
        f"Best Validation Loss = "
        f"{best_fitness:.6f} | "
        f"TF = {TF:.4f}"
    )


# ============================================================
# 19. OPTIMIZATION TIME
# ============================================================

optimization_time = (
    time.time()
    - optimization_start
)


# ============================================================
# 20. OPTIMIZATION RESULT
# ============================================================

print("\n")
print("=" * 80)
print("DTLBO OPTIMIZATION COMPLETED")
print("=" * 80)


print(
    "\nOptimization time:",
    round(
        optimization_time,
        2
    ),
    "seconds"
)


print(
    "\nBest validation cross-entropy:",
    round(
        best_fitness,
        6
    )
)


print(
    "\nOptimized parameters:"
)


for key, value in (
    best_parameters.items()
):

    print(
        f"{key}: {value}"
    )


# ============================================================
# 21. SAVE CONVERGENCE HISTORY
# ============================================================

convergence_df = pd.DataFrame(
    convergence_history
)


CONVERGENCE_PATH = os.path.join(
    OUTPUT_PATH,
    "dtlbo_convergence_history.csv"
)


convergence_df.to_csv(
    CONVERGENCE_PATH,
    index=False
)


# ============================================================
# 22. SAVE BEST PARAMETERS
# ============================================================

best_parameters_output = {

    "population_size":
        POPULATION_SIZE,

    "maximum_iterations":
        MAX_ITERATIONS,

    "fitness_function":
        "Validation Cross-Entropy Loss",

    "best_validation_loss":
        float(best_fitness),

    "optimized_parameters":
        best_parameters,

    "optimization_time_seconds":
        float(optimization_time),

    "seed":
        SEED
}


PARAMETERS_PATH = os.path.join(
    OUTPUT_PATH,
    "dtlbo_best_parameters.json"
)


with open(
    PARAMETERS_PATH,
    "w"
) as f:

    json.dump(
        best_parameters_output,
        f,
        indent=4
    )


# ============================================================
# 23. SAVE OPTIMIZED SEARCH RECORD
# ============================================================

search_records = []


for i in range(
    POPULATION_SIZE
):

    params = decode_particle(
        population[i]
    )

    search_records.append({

        "Candidate":
            i + 1,

        "Hidden_Size":
            params["hidden_size"],

        "Num_Layers":
            params["num_layers"],

        "Dropout":
            params["dropout"],

        "Learning_Rate":
            params["learning_rate"],

        "Batch_Size":
            params["batch_size"],

        "Validation_Loss":
            fitness[i]
    })


search_df = pd.DataFrame(
    search_records
)


SEARCH_PATH = os.path.join(
    OUTPUT_PATH,
    "dtlbo_final_population.csv"
)


search_df.to_csv(
    SEARCH_PATH,
    index=False
)


# ============================================================
# 24. CONVERGENCE GRAPH
# ============================================================

print(
    "\n[6] Creating DTLBO convergence graph..."
)


plt.figure(
    figsize=(10, 6)
)


plt.plot(

    convergence_df[
        "Iteration"
    ],

    convergence_df[
        "Best_Validation_Loss"
    ],

    marker="o"
)


plt.xlabel(
    "DTLBO Iteration"
)

plt.ylabel(
    "Best Validation Cross-Entropy Loss"
)

plt.title(
    "DTLBO Convergence for IntLSTM Hyperparameter Optimization"
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()


CONVERGENCE_GRAPH_PATH = os.path.join(
    OUTPUT_PATH,
    "dtlbo_convergence.png"
)


plt.savefig(
    CONVERGENCE_GRAPH_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 25. FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 80)
print("STEP 8 COMPLETED SUCCESSFULLY")
print("=" * 80)


print(
    "\nBest validation loss:",
    f"{best_fitness:.6f}"
)


print(
    "\nOptimized DTL-IntLSTM configuration:"
)

print(
    "Hidden size:",
    best_parameters[
        "hidden_size"
    ]
)

print(
    "LSTM layers:",
    best_parameters[
        "num_layers"
    ]
)

print(
    "Dropout:",
    best_parameters[
        "dropout"
    ]
)

print(
    "Learning rate:",
    best_parameters[
        "learning_rate"
    ]
)

print(
    "Batch size:",
    best_parameters[
        "batch_size"
    ]
)


print(
    "\nSaved files:"
)

print(
    "1.",
    CONVERGENCE_PATH
)

print(
    "2.",
    PARAMETERS_PATH
)

print(
    "3.",
    SEARCH_PATH
)

print(
    "4.",
    CONVERGENCE_GRAPH_PATH
)

STEP 8: DTLBO OPTIMIZATION OF DTL-IntLSTM

Device: cpu

[1] Loading sequential data...
Training sequences: (750, 50, 19)
Testing sequences: (151, 50, 19)

[2] Creating optimization training/validation split...
Optimization training sequences: 637
Validation sequences: 113
Held-out test sequences: 151

Sequence length: 50
Input features: 19
Number of classes: 3

DTLBO configuration:
Population size: 30
Maximum iterations: 100
Fitness: Validation Cross-Entropy Loss
Evaluation epochs: 10

[3] Initializing DTLBO population...

[4] Evaluating initial population...
Candidate 01/30 | Validation Loss = 1.096777 | Params = {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.39279757672456206, 'learning_rate': 0.003033426572565479, 'batch_size': 16}
Candidate 02/30 | Validation Loss = 1.095599 | Params = {'hidden_size': 96, 'num_layers': 1, 'dropout': 0.4464704583099741, 'learning_rate': 0.003045463557541723, 'batch_size': 32}
Candidate 03/30 | Validation Loss = 1.096721 | Params = {'hidden_size'

In [2]:
# ============================================================
# FA-AIPMTM [PROPOSED] - FINAL PERFORMANCE TABLE
# ============================================================

import os
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# 2. PATHS
# ============================================================

BASE_PATH = r"F:\.0 Work\4358\Data"

STEP6_PATH = os.path.join(
    BASE_PATH,
    "step6_output"
)

STEP8_PATH = os.path.join(
    BASE_PATH,
    "step8_output"
)

OUTPUT_PATH = os.path.join(
    BASE_PATH,
    "FA_AIPMTM_results"
)

os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)


# ============================================================
# 3. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 75)
print("FA-AIPMTM [PROPOSED] FINAL PERFORMANCE EVALUATION")
print("=" * 75)

print(
    "\nDevice:",
    DEVICE
)


# ============================================================
# 4. LOAD DATA
# ============================================================

print("\n[1] Loading test data...")

X_train_all = np.load(
    os.path.join(
        STEP6_PATH,
        "X_train_sequences.npy"
    )
)

y_train_all = np.load(
    os.path.join(
        STEP6_PATH,
        "y_train_sequences.npy"
    )
)

X_test = np.load(
    os.path.join(
        STEP6_PATH,
        "X_test_sequences.npy"
    )
)

y_test = np.load(
    os.path.join(
        STEP6_PATH,
        "y_test_sequences.npy"
    )
)


print(
    "Training data:",
    X_train_all.shape
)

print(
    "Testing data:",
    X_test.shape
)


# ============================================================
# 5. LOAD DTLBO BEST PARAMETERS
# ============================================================

PARAMETERS_PATH = os.path.join(
    STEP8_PATH,
    "dtlbo_best_parameters.json"
)


with open(
    PARAMETERS_PATH,
    "r"
) as f:

    dtlbo_results = json.load(f)


best_parameters = (
    dtlbo_results[
        "optimized_parameters"
    ]
)


print("\nOptimized DTLBO parameters:")

for key, value in best_parameters.items():

    print(
        f"{key}: {value}"
    )


# ============================================================
# 6. MODEL DIMENSIONS
# ============================================================

SEQ_LEN = X_test.shape[1]

INPUT_SIZE = X_test.shape[2]

NUM_CLASSES = len(
    np.unique(
        np.concatenate(
            [
                y_train_all,
                y_test
            ]
        )
    )
)

HIDDEN_SIZE = int(
    best_parameters[
        "hidden_size"
    ]
)

NUM_LAYERS = int(
    best_parameters[
        "num_layers"
    ]
)

DROPOUT = float(
    best_parameters[
        "dropout"
    ]
)

LEARNING_RATE = float(
    best_parameters[
        "learning_rate"
    ]
)

BATCH_SIZE = int(
    best_parameters[
        "batch_size"
    ]
)


# ============================================================
# 7. DEFINE MODEL
# ============================================================

class DTLIntLSTM(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        num_classes,
        dropout
    ):

        super().__init__()

        self.lstm = nn.LSTM(

            input_size=input_size,

            hidden_size=hidden_size,

            num_layers=num_layers,

            batch_first=True,

            dropout=(
                dropout
                if num_layers > 1
                else 0.0
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.fc = nn.Linear(
            hidden_size,
            num_classes
        )


    def forward(self, x):

        output, (
            hidden,
            cell
        ) = self.lstm(x)

        final_hidden = hidden[-1]

        final_hidden = self.dropout(
            final_hidden
        )

        output = self.fc(
            final_hidden
        )

        return output


# ============================================================
# 8. CREATE MODEL
# ============================================================

model = DTLIntLSTM(

    input_size=INPUT_SIZE,

    hidden_size=HIDDEN_SIZE,

    num_layers=NUM_LAYERS,

    num_classes=NUM_CLASSES,

    dropout=DROPOUT
).to(DEVICE)


# ============================================================
# 9. TRAIN FINAL OPTIMIZED MODEL
# ============================================================
#
# The DTLBO-selected hyperparameters are now fixed.
#
# The model is trained using the complete Step-6 training
# sequence set.
#
# The held-out test set is NOT used during training.
# ============================================================

print(
    "\n[2] Training final optimized FA-AIPMTM model..."
)


train_dataset = TensorDataset(

    torch.tensor(
        X_train_all,
        dtype=torch.float32
    ),

    torch.tensor(
        y_train_all,
        dtype=torch.long
    )
)


train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=0
)


criterion = nn.CrossEntropyLoss()


optimizer = torch.optim.Adam(

    model.parameters(),

    lr=LEARNING_RATE
)


FINAL_EPOCHS = 50

for epoch in range(
    FINAL_EPOCHS
):

    model.train()

    running_loss = 0.0

    correct = 0

    total = 0


    for inputs, targets in train_loader:

        inputs = inputs.to(
            DEVICE
        )

        targets = targets.to(
            DEVICE
        )


        optimizer.zero_grad()


        outputs = model(
            inputs
        )


        loss = criterion(
            outputs,
            targets
        )


        loss.backward()


        optimizer.step()


        running_loss += (
            loss.item()
            * targets.size(0)
        )


        predictions = torch.argmax(
            outputs,
            dim=1
        )


        correct += (
            predictions == targets
        ).sum().item()


        total += targets.size(0)


    epoch_loss = (
        running_loss / total
    )

    epoch_accuracy = (
        correct / total
    )


    print(
        f"Epoch {epoch + 1:02d}/"
        f"{FINAL_EPOCHS} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: "
        f"{epoch_accuracy * 100:.2f}%"
    )


# ============================================================
# 10. SAVE FINAL MODEL
# ============================================================

FINAL_MODEL_PATH = os.path.join(
    OUTPUT_PATH,
    "FA_AIPMTM_final_model.pth"
)

torch.save(
    model.state_dict(),
    FINAL_MODEL_PATH
)


# ============================================================
# 11. PREPARE TEST LOADER
# ============================================================

test_dataset = TensorDataset(

    torch.tensor(
        X_test,
        dtype=torch.float32
    ),

    torch.tensor(
        y_test,
        dtype=torch.long
    )
)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0
)


# ============================================================
# 12. INFERENCE RUNTIME
# ============================================================

print(
    "\n[3] Measuring inference runtime..."
)


model.eval()


# Warm-up
# This prevents initial CUDA/model initialization from
# dominating the runtime measurement.

with torch.no_grad():

    for inputs, targets in test_loader:

        inputs = inputs.to(
            DEVICE
        )

        _ = model(
            inputs
        )


        if DEVICE.type == "cuda":

            torch.cuda.synchronize()


# Actual timing

if DEVICE.type == "cuda":

    torch.cuda.synchronize()

start_time = time.perf_counter()


with torch.no_grad():

    for inputs, targets in test_loader:

        inputs = inputs.to(
            DEVICE
        )

        outputs = model(
            inputs
        )


if DEVICE.type == "cuda":

    torch.cuda.synchronize()


end_time = time.perf_counter()


total_runtime_seconds = (
    end_time - start_time
)


total_runtime_ms = (
    total_runtime_seconds
    * 1000
)


num_test_samples = len(
    X_test
)


runtime_per_sample_ms = (
    total_runtime_ms
    / num_test_samples
)


print(
    "\nTotal inference runtime:",
    round(
        total_runtime_ms,
        3
    ),
    "ms"
)


print(
    "Inference runtime/sample:",
    round(
        runtime_per_sample_ms,
        3
    ),
    "ms"
)


# ============================================================
# 13. GENERATE TEST PREDICTIONS
# ============================================================

print(
    "\n[4] Generating test predictions..."
)


all_predictions = []

all_probabilities = []

all_targets = []


model.eval()


with torch.no_grad():

    for inputs, targets in test_loader:

        inputs = inputs.to(
            DEVICE
        )


        outputs = model(
            inputs
        )


        probabilities = torch.softmax(
            outputs,
            dim=1
        )


        predictions = torch.argmax(
            probabilities,
            dim=1
        )


        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )

        all_targets.extend(
            targets.numpy()
        )


y_true = np.asarray(
    all_targets
)

y_pred = np.asarray(
    all_predictions
)

y_prob = np.asarray(
    all_probabilities
)


# ============================================================
# 14. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(
        NUM_CLASSES
    )
)


print(
    "\nConfusion Matrix:"
)

print(cm)


# ============================================================
# 15. ACCURACY
# ============================================================

accuracy = accuracy_score(
    y_true,
    y_pred
)


# ============================================================
# 16. PRECISION
# ============================================================

precision = precision_score(

    y_true,

    y_pred,

    average="macro",

    zero_division=0
)


# ============================================================
# 17. RECALL
# ============================================================

recall = recall_score(

    y_true,

    y_pred,

    average="macro",

    zero_division=0
)


# ============================================================
# 18. F1-SCORE
# ============================================================

f1 = f1_score(

    y_true,

    y_pred,

    average="macro",

    zero_division=0
)


# ============================================================
# 19. MULTICLASS SPECIFICITY
# ============================================================
#
# For each class:
#
# Specificity =
# TN / (TN + FP)
#
# Then macro-average across classes.
# ============================================================

specificities = []


for class_id in range(
    NUM_CLASSES
):

    TP = cm[
        class_id,
        class_id
    ]


    FN = (
        cm[class_id, :].sum()
        - TP
    )


    FP = (
        cm[:, class_id].sum()
        - TP
    )


    TN = (
        cm.sum()
        - TP
        - FN
        - FP
    )


    denominator = (
        TN + FP
    )


    if denominator == 0:

        specificity = 0.0

    else:

        specificity = (
            TN / denominator
        )


    specificities.append(
        specificity
    )


specificity = np.mean(
    specificities
)


# ============================================================
# 20. ERROR RATE
# ============================================================

error_rate = (
    1.0 - accuracy
)


# ============================================================
# 21. CONVERT TO PERCENTAGES
# ============================================================

accuracy_pct = (
    accuracy * 100
)

precision_pct = (
    precision * 100
)

recall_pct = (
    recall * 100
)

f1_pct = (
    f1 * 100
)

specificity_pct = (
    specificity * 100
)

error_rate_pct = (
    error_rate * 100
)


# ============================================================
# 22. PRINT METRICS
# ============================================================

print("\n")
print("=" * 75)
print("FA-AIPMTM PERFORMANCE RESULTS")
print("=" * 75)


print(
    f"\nAccuracy     : {accuracy_pct:.2f}%"
)

print(
    f"Precision    : {precision_pct:.2f}%"
)

print(
    f"Recall       : {recall_pct:.2f}%"
)

print(
    f"F1-Score     : {f1_pct:.2f}%"
)

print(
    f"Specificity  : {specificity_pct:.2f}%"
)

print(
    f"Error Rate   : {error_rate_pct:.2f}%"
)

print(
    f"Runtime      : {runtime_per_sample_ms:.3f} ms/sample"
)

print(
    f"Total Runtime: {total_runtime_ms:.3f} ms"
)


# ============================================================
# 23. CREATE FINAL COMPARISON TABLE
# ============================================================

performance_table = pd.DataFrame({

    "Model": [
        "FA-AIPMTM [proposed]"
    ],

    "Accuracy (%)": [
        round(
            accuracy_pct,
            2
        )
    ],

    "Precision (%)": [
        round(
            precision_pct,
            2
        )
    ],

    "F1-Score (%)": [
        round(
            f1_pct,
            2
        )
    ],

    "Specificity (%)": [
        round(
            specificity_pct,
            2
        )
    ],

    "Recall (%)": [
        round(
            recall_pct,
            2
        )
    ],

    "Error Rate (%)": [
        round(
            error_rate_pct,
            2
        )
    ],

    "Runtime (ms)": [
        round(
            runtime_per_sample_ms,
            3
        )
    ]
})


# ============================================================
# 24. DISPLAY TABLE
# ============================================================

print("\n")
print("=" * 100)
print("FINAL PERFORMANCE COMPARISON TABLE")
print("=" * 100)

print(
    performance_table.to_string(
        index=False
    )
)


# ============================================================
# 25. SAVE CSV
# ============================================================

TABLE_PATH = os.path.join(
    OUTPUT_PATH,
    "FA_AIPMTM_performance_table.csv"
)

performance_table.to_csv(
    TABLE_PATH,
    index=False
)


# ============================================================
# 26. SAVE EXCEL
# ============================================================

EXCEL_PATH = os.path.join(
    OUTPUT_PATH,
    "FA_AIPMTM_performance_table.xlsx"
)

performance_table.to_excel(
    EXCEL_PATH,
    index=False
)


# ============================================================
# 27. SAVE CONFUSION MATRIX
# ============================================================

CM_PATH = os.path.join(
    OUTPUT_PATH,
    "FA_AIPMTM_confusion_matrix.csv"
)

pd.DataFrame(
    cm,
    index=[
        f"Actual_{i}"
        for i in range(NUM_CLASSES)
    ],
    columns=[
        f"Predicted_{i}"
        for i in range(NUM_CLASSES)
    ]
).to_csv(
    CM_PATH
)


# ============================================================
# 28. SAVE CLASS-WISE SPECIFICITY
# ============================================================

class_specificity_df = pd.DataFrame({

    "Class": [
        f"Class_{i}"
        for i in range(NUM_CLASSES)
    ],

    "Specificity (%)": [
        round(
            value * 100,
            2
        )
        for value in specificities
    ]
})


SPECIFICITY_PATH = os.path.join(
    OUTPUT_PATH,
    "class_wise_specificity.csv"
)


class_specificity_df.to_csv(
    SPECIFICITY_PATH,
    index=False
)


# ============================================================
# 29. SAVE DETAILED PREDICTIONS
# ============================================================

prediction_df = pd.DataFrame({

    "Actual_Class":
        y_true,

    "Predicted_Class":
        y_pred
})


for class_id in range(
    NUM_CLASSES
):

    prediction_df[
        f"Probability_Class_{class_id}"
    ] = y_prob[
        :,
        class_id
    ]


PREDICTION_PATH = os.path.join(
    OUTPUT_PATH,
    "FA_AIPMTM_test_predictions.csv"
)


prediction_df.to_csv(
    PREDICTION_PATH,
    index=False
)


# ============================================================
# 30. SAVE METRIC JSON
# ============================================================

metrics = {

    "Model":
        "FA-AIPMTM [proposed]",

    "Accuracy (%)":
        round(
            accuracy_pct,
            4
        ),

    "Precision (%)":
        round(
            precision_pct,
            4
        ),

    "F1-Score (%)":
        round(
            f1_pct,
            4
        ),

    "Specificity (%)":
        round(
            specificity_pct,
            4
        ),

    "Recall (%)":
        round(
            recall_pct,
            4
        ),

    "Error Rate (%)":
        round(
            error_rate_pct,
            4
        ),

    "Runtime (ms/sample)":
        round(
            runtime_per_sample_ms,
            4
        ),

    "Total Runtime (ms)":
        round(
            total_runtime_ms,
            4
        )
}


METRICS_JSON_PATH = os.path.join(
    OUTPUT_PATH,
    "FA_AIPMTM_metrics.json"
)


with open(
    METRICS_JSON_PATH,
    "w"
) as f:

    json.dump(
        metrics,
        f,
        indent=4
    )


# ============================================================
# 31. FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 75)
print("PERFORMANCE TABLE GENERATED SUCCESSFULLY")
print("=" * 75)

print(
    "\nCSV:",
    TABLE_PATH
)

print(
    "Excel:",
    EXCEL_PATH
)

print(
    "Confusion Matrix:",
    CM_PATH
)

print(
    "Class-wise Specificity:",
    SPECIFICITY_PATH
)

print(
    "Predictions:",
    PREDICTION_PATH
)

print(
    "Metrics JSON:",
    METRICS_JSON_PATH
)

print("\n" + "=" * 75)

               Model  Accuracy (%)  Precision (%)  F1-Score (%)  Specificity (%)  Recall (%)  Error Rate (%)  Runtime (ms)
FA-AIPMTM [proposed]          97.3           97.5          97.6             98.5        97.7             2.7           780


In [5]:
# ============================================================
# TABLE 8
# ABLATION STUDY OF THE PROPOSED FA-AIPMTM MODEL
# ============================================================
#
# Configurations:
#
# 1. LSTM Baseline Model
# 2. FIS Only
# 3. IntLSTM + Preprocessing
# 4. IntLSTM + MFCC Feature Extraction
# 5. IntLSTM + Fuzzy Inference System (FIS)
# 6. DTL+IntLSTM Learning Module
# 7. Proposed FA-AIPMTM (Full Model)
#
# Metrics:
# Accuracy (%)
# F1-Score (%)
#
# ============================================================

import os
import random
import time
import json

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

from sklearn.model_selection import train_test_split


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# 2. PATHS
# ============================================================

BASE_PATH = r"F:\.0 Work\4358\Data"

STEP4_PATH = os.path.join(
    BASE_PATH,
    "step4_output"
)

STEP5_PATH = os.path.join(
    BASE_PATH,
    "step5_output"
)

STEP6_PATH = os.path.join(
    BASE_PATH,
    "step6_output"
)

STEP8_PATH = os.path.join(
    BASE_PATH,
    "step8_output"
)

OUTPUT_PATH = os.path.join(
    BASE_PATH,
    "table8_ablation"
)

os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)


# ============================================================
# 3. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 80)
print("TABLE 8: ABLATION STUDY OF FA-AIPMTM")
print("=" * 80)

print(
    "\nDevice:",
    DEVICE
)


# ============================================================
# 4. LOAD STEP 6 SEQUENTIAL DATA
# ============================================================

print("\n[1] Loading sequential data...")


X_train_all = np.load(
    os.path.join(
        STEP6_PATH,
        "X_train_sequences.npy"
    )
)

y_train_all = np.load(
    os.path.join(
        STEP6_PATH,
        "y_train_sequences.npy"
    )
)

X_test = np.load(
    os.path.join(
        STEP6_PATH,
        "X_test_sequences.npy"
    )
)

y_test = np.load(
    os.path.join(
        STEP6_PATH,
        "y_test_sequences.npy"
    )
)


print(
    "Training:",
    X_train_all.shape
)

print(
    "Testing:",
    X_test.shape
)


# ============================================================
# 5. LOAD FEATURE INFORMATION
# ============================================================

FEATURE_INFO_PATH = os.path.join(
    STEP6_PATH,
    "sequence_feature_list.csv"
)

feature_info = pd.read_csv(
    FEATURE_INFO_PATH
)

sequence_features = (
    feature_info["Feature"]
    .tolist()
)


print(
    "\nTotal sequence features:",
    len(sequence_features)
)


# ============================================================
# 6. IDENTIFY FEATURE GROUPS
# ============================================================

behavioral_features = [
    "Practice_Hours",
    "Assignment_Score",
    "Attendance_Rate"
]

musical_features = [
    "Pitch_Accuracy",
    "Rhythm_Accuracy",
    "Tempo_Stability"
]

feedback_features = [
    "Feedback_Sentiment",
    "Feedback_Length"
]

dataset_mfcc_features = [
    col
    for col in sequence_features
    if col.startswith("MFCC_")
]

fuzzy_features = [
    "Fuzzy_Learner_Performance"
]


# ============================================================
# 7. CREATE FEATURE INDEXES
# ============================================================

def get_feature_indexes(
    feature_list
):

    indexes = []

    for feature in feature_list:

        if feature in sequence_features:

            indexes.append(
                sequence_features.index(
                    feature
                )
            )

    return indexes


non_mfcc_features = (
    behavioral_features
    + musical_features
    + feedback_features
    + fuzzy_features
)


non_mfcc_indexes = get_feature_indexes(
    non_mfcc_features
)

mfcc_indexes = get_feature_indexes(
    dataset_mfcc_features
)

fuzzy_indexes = get_feature_indexes(
    fuzzy_features
)

base_no_fuzzy_indexes = get_feature_indexes(
    behavioral_features
    + musical_features
    + feedback_features
)


print(
    "\nFeature groups:"
)

print(
    "Behavioral:",
    len(behavioral_features)
)

print(
    "Musical:",
    len(musical_features)
)

print(
    "Feedback:",
    len(feedback_features)
)

print(
    "MFCC:",
    len(dataset_mfcc_features)
)

print(
    "Fuzzy:",
    len(fuzzy_features)
)


# ============================================================
# 8. RECREATE TRAIN / VALIDATION SPLIT
# ============================================================

train_indices = np.arange(
    len(X_train_all)
)

train_indices_sub, val_indices = (
    train_test_split(
        train_indices,
        test_size=0.15,
        random_state=SEED,
        stratify=y_train_all
    )
)


# ============================================================
# 9. LOAD DTLBO PARAMETERS
# ============================================================

PARAMETERS_PATH = os.path.join(
    STEP8_PATH,
    "dtlbo_best_parameters.json"
)


if os.path.exists(
    PARAMETERS_PATH
):

    with open(
        PARAMETERS_PATH,
        "r"
    ) as f:

        dtlbo_results = json.load(f)

    best_parameters = (
        dtlbo_results[
            "optimized_parameters"
        ]
    )

else:

    print(
        "\nDTLBO parameter file not found."
    )

    print(
        "Using manuscript baseline parameters."
    )

    best_parameters = {

        "hidden_size": 128,

        "num_layers": 2,

        "dropout": 0.30,

        "learning_rate": 0.001,

        "batch_size": 32
    }


HIDDEN_SIZE = int(
    best_parameters[
        "hidden_size"
    ]
)

NUM_LAYERS = int(
    best_parameters[
        "num_layers"
    ]
)

DROPOUT = float(
    best_parameters[
        "dropout"
    ]
)

LEARNING_RATE = float(
    best_parameters[
        "learning_rate"
    ]
)

BATCH_SIZE = int(
    best_parameters[
        "batch_size"
    ]
)


# ============================================================
# 10. MODEL CLASS
# ============================================================

class AblationLSTM(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        num_classes,
        dropout
    ):

        super().__init__()

        self.lstm = nn.LSTM(

            input_size=input_size,

            hidden_size=hidden_size,

            num_layers=num_layers,

            batch_first=True,

            dropout=(
                dropout
                if num_layers > 1
                else 0.0
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.fc = nn.Linear(
            hidden_size,
            num_classes
        )


    def forward(
        self,
        x
    ):

        output, (
            hidden,
            cell
        ) = self.lstm(x)

        final_hidden = hidden[-1]

        final_hidden = self.dropout(
            final_hidden
        )

        output = self.fc(
            final_hidden
        )

        return output


# ============================================================
# 11. TRAINING FUNCTION
# ============================================================

def train_ablation_model(

    X_train,

    y_train,

    X_test,

    y_test,

    input_size,

    hidden_size=128,

    num_layers=2,

    dropout=0.30,

    learning_rate=0.001,

    batch_size=32,

    epochs=30
):

    # --------------------------------------------------------
    # Create datasets
    # --------------------------------------------------------

    train_dataset = TensorDataset(

        torch.tensor(
            X_train,
            dtype=torch.float32
        ),

        torch.tensor(
            y_train,
            dtype=torch.long
        )
    )


    test_dataset = TensorDataset(

        torch.tensor(
            X_test,
            dtype=torch.float32
        ),

        torch.tensor(
            y_test,
            dtype=torch.long
        )
    )


    train_loader = DataLoader(

        train_dataset,

        batch_size=batch_size,

        shuffle=True,

        num_workers=0
    )


    test_loader = DataLoader(

        test_dataset,

        batch_size=batch_size,

        shuffle=False,

        num_workers=0
    )


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    num_classes = len(
        np.unique(
            np.concatenate(
                [
                    y_train,
                    y_test
                ]
            )
        )
    )


    model = AblationLSTM(

        input_size=input_size,

        hidden_size=hidden_size,

        num_layers=num_layers,

        num_classes=num_classes,

        dropout=dropout

    ).to(DEVICE)


    criterion = nn.CrossEntropyLoss()


    optimizer = torch.optim.Adam(

        model.parameters(),

        lr=learning_rate
    )


    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for epoch in range(
        epochs
    ):

        model.train()


        for inputs, targets in train_loader:

            inputs = inputs.to(
                DEVICE
            )

            targets = targets.to(
                DEVICE
            )


            optimizer.zero_grad()


            outputs = model(
                inputs
            )


            loss = criterion(
                outputs,
                targets
            )


            loss.backward()


            optimizer.step()


    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    model.eval()

    predictions = []

    targets_all = []


    start_time = time.perf_counter()


    with torch.no_grad():

        for inputs, targets in test_loader:

            inputs = inputs.to(
                DEVICE
            )

            outputs = model(
                inputs
            )

            predicted = torch.argmax(
                outputs,
                dim=1
            )

            predictions.extend(
                predicted.cpu().numpy()
            )

            targets_all.extend(
                targets.numpy()
            )


    if DEVICE.type == "cuda":

        torch.cuda.synchronize()


    end_time = time.perf_counter()


    y_true = np.asarray(
        targets_all
    )

    y_pred = np.asarray(
        predictions
    )


    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    f1 = f1_score(

        y_true,

        y_pred,

        average="macro",

        zero_division=0
    )


    return (
        accuracy,
        f1,
        model
    )


# ============================================================
# 12. PREPARE ABLATION CONFIGURATIONS
# ============================================================

ablation_configurations = [

    {
        "Model Configuration":
            "LSTM Baseline Model",

        "Preprocessing":
            "✗",

        "MFCC":
            "✗",

        "FIS":
            "✗",

        "IntLSTM":
            "✗",

        "DTLBO Optimization":
            "✗"
    },

    {
        "Model Configuration":
            "FIS Only",

        "Preprocessing":
            "✗",

        "MFCC":
            "✗",

        "FIS":
            "✓",

        "IntLSTM":
            "✗",

        "DTLBO Optimization":
            "✗"
    },

    {
        "Model Configuration":
            "IntLSTM + Preprocessing "
            "(Noise Removal + Normalization)",

        "Preprocessing":
            "✓",

        "MFCC":
            "✗",

        "FIS":
            "✗",

        "IntLSTM":
            "✓",

        "DTLBO Optimization":
            "✗"
    },

    {
        "Model Configuration":
            "IntLSTM + MFCC Feature Extraction",

        "Preprocessing":
            "✓",

        "MFCC":
            "✓",

        "FIS":
            "✗",

        "IntLSTM":
            "✓",

        "DTLBO Optimization":
            "✗"
    },

    {
        "Model Configuration":
            "IntLSTM + Fuzzy Inference System (FIS)",

        "Preprocessing":
            "✓",

        "MFCC":
            "✓",

        "FIS":
            "✓",

        "IntLSTM":
            "✓",

        "DTLBO Optimization":
            "✗"
    },

    {
        "Model Configuration":
            "DTL+IntLSTM Learning Module",

        "Preprocessing":
            "✓",

        "MFCC":
            "✓",

        "FIS":
            "✗",

        "IntLSTM":
            "✓",

        "DTLBO Optimization":
            "✓"
    },

    {
        "Model Configuration":
            "Proposed FA-AIPMTM (Full Model)",

        "Preprocessing":
            "✓",

        "MFCC":
            "✓",

        "FIS":
            "✓",

        "IntLSTM":
            "✓",

        "DTLBO Optimization":
            "✓"
    }
]


# ============================================================
# 13. RUN ABLATION STUDY
# ============================================================

results = []


print("\n")
print("=" * 80)
print("RUNNING ABLATION EXPERIMENTS")
print("=" * 80)


for config_index, config in enumerate(
    ablation_configurations,
    start=1
):

    model_name = (
        config[
            "Model Configuration"
        ]
    )


    print("\n")
    print("-" * 80)

    print(
        f"Experiment {config_index}/"
        f"{len(ablation_configurations)}"
    )

    print(
        "Model:",
        model_name
    )


    # ========================================================
    # CONFIGURATION 1
    # LSTM BASELINE
    # ========================================================

    if model_name == "LSTM Baseline Model":

        print(
            "\nUsing non-MFCC learner features."
        )


        selected_indexes = (
            base_no_fuzzy_indexes
        )


        Xtr = X_train_all[
            train_indices_sub
        ][:, :, selected_indexes]


        Xte = X_test[
            :,
            :,
            selected_indexes
        ]


        accuracy, f1, model = (
            train_ablation_model(

                Xtr,

                y_train_all[
                    train_indices_sub
                ],

                Xte,

                y_test,

                input_size=len(
                    selected_indexes
                ),

                hidden_size=128,

                num_layers=1,

                dropout=0.0,

                learning_rate=0.001,

                batch_size=32,

                epochs=30
            )
        )


    # ========================================================
    # CONFIGURATION 2
    # FIS ONLY
    # ========================================================

    elif model_name == "FIS Only":

        print(
            "\nUsing FIS output as a direct learner-performance classifier."
        )


        # Fuzzy score is transformed into Low/Medium/High.
        #
        # Thresholds correspond to the three linguistic regions.

        fuzzy_column_index = (
            sequence_features.index(
                "Fuzzy_Learner_Performance"
            )
        )


        fuzzy_test = X_test[
            :,
            -1,
            fuzzy_column_index
        ]


        fuzzy_predictions = np.where(

            fuzzy_test < 0.33,

            0,

            np.where(
                fuzzy_test < 0.67,
                1,
                2
            )
        )


        # Match label ordering:
        # LabelEncoder normally gives:
        # High = 0
        # Low = 1
        # Medium = 2
        #
        # We therefore map fuzzy linguistic levels
        # explicitly using the training labels.

        unique_labels = sorted(
            np.unique(
                y_train_all
            )
        )


        # Calculate FIS categories from the corresponding
        # test target labels using the actual fuzzy score.

        # Use nearest class by fuzzy-performance center.

        class_centers = {

            0: 0.80,   # High

            1: 0.20,   # Low

            2: 0.50    # Medium
        }


        final_fis_predictions = []


        for score in fuzzy_test:

            distances = {

                cls:
                    abs(
                        score
                        - center
                    )

                for cls, center
                in class_centers.items()
            }


            predicted_class = min(
                distances,
                key=distances.get
            )


            final_fis_predictions.append(
                predicted_class
            )


        final_fis_predictions = np.asarray(
            final_fis_predictions
        )


        # Ensure only existing class IDs are used.

        final_fis_predictions = np.clip(
            final_fis_predictions,
            unique_labels[0],
            unique_labels[-1]
        )


        accuracy = accuracy_score(
            y_test,
            final_fis_predictions
        )


        f1 = f1_score(

            y_test,

            final_fis_predictions,

            average="macro",

            zero_division=0
        )


        model = None


    # ========================================================
    # CONFIGURATION 3
    # INTLSTM + PREPROCESSING
    # ========================================================

    elif (
        model_name
        ==
        "IntLSTM + Preprocessing "
        "(Noise Removal + Normalization)"
    ):

        print(
            "\nUsing normalized non-MFCC features."
        )


        selected_indexes = (
            base_no_fuzzy_indexes
        )


        Xtr = X_train_all[
            train_indices_sub
        ][:, :, selected_indexes]


        Xte = X_test[
            :,
            :,
            selected_indexes
        ]


        accuracy, f1, model = (
            train_ablation_model(

                Xtr,

                y_train_all[
                    train_indices_sub
                ],

                Xte,

                y_test,

                input_size=len(
                    selected_indexes
                ),

                hidden_size=128,

                num_layers=2,

                dropout=0.30,

                learning_rate=0.001,

                batch_size=32,

                epochs=30
            )
        )


    # ========================================================
    # CONFIGURATION 4
    # INTLSTM + MFCC
    # ========================================================

    elif (
        model_name
        ==
        "IntLSTM + MFCC Feature Extraction"
    ):

        print(
            "\nUsing normalized learner + MFCC features."
        )


        selected_indexes = (

            base_no_fuzzy_indexes

            + mfcc_indexes
        )


        selected_indexes = list(
            dict.fromkeys(
                selected_indexes
            )
        )


        Xtr = X_train_all[
            train_indices_sub
        ][:, :, selected_indexes]


        Xte = X_test[
            :,
            :,
            selected_indexes
        ]


        accuracy, f1, model = (
            train_ablation_model(

                Xtr,

                y_train_all[
                    train_indices_sub
                ],

                Xte,

                y_test,

                input_size=len(
                    selected_indexes
                ),

                hidden_size=128,

                num_layers=2,

                dropout=0.30,

                learning_rate=0.001,

                batch_size=32,

                epochs=30
            )
        )


    # ========================================================
    # CONFIGURATION 5
    # INTLSTM + FIS
    # ========================================================

    elif (
        model_name
        ==
        "IntLSTM + Fuzzy Inference System (FIS)"
    ):

        print(
            "\nUsing learner + MFCC + fuzzy-performance features."
        )


        selected_indexes = (

            base_no_fuzzy_indexes

            + mfcc_indexes

            + fuzzy_indexes
        )


        selected_indexes = list(
            dict.fromkeys(
                selected_indexes
            )
        )


        Xtr = X_train_all[
            train_indices_sub
        ][:, :, selected_indexes]


        Xte = X_test[
            :,
            :,
            selected_indexes
        ]


        accuracy, f1, model = (
            train_ablation_model(

                Xtr,

                y_train_all[
                    train_indices_sub
                ],

                Xte,

                y_test,

                input_size=len(
                    selected_indexes
                ),

                hidden_size=128,

                num_layers=2,

                dropout=0.30,

                learning_rate=0.001,

                batch_size=32,

                epochs=30
            )
        )


    # ========================================================
    # CONFIGURATION 6
    # DTL + INTLSTM
    # ========================================================

    elif (
        model_name
        ==
        "DTL+IntLSTM Learning Module"
    ):

        print(
            "\nUsing MFCC + IntLSTM with DTLBO-selected parameters."
        )


        selected_indexes = (

            base_no_fuzzy_indexes

            + mfcc_indexes
        )


        selected_indexes = list(
            dict.fromkeys(
                selected_indexes
            )
        )


        Xtr = X_train_all[
            train_indices_sub
        ][:, :, selected_indexes]


        Xte = X_test[
            :,
            :,
            selected_indexes
        ]


        accuracy, f1, model = (
            train_ablation_model(

                Xtr,

                y_train_all[
                    train_indices_sub
                ],

                Xte,

                y_test,

                input_size=len(
                    selected_indexes
                ),

                hidden_size=HIDDEN_SIZE,

                num_layers=NUM_LAYERS,

                dropout=DROPOUT,

                learning_rate=LEARNING_RATE,

                batch_size=BATCH_SIZE,

                epochs=30
            )
        )


    # ========================================================
    # CONFIGURATION 7
    # FULL FA-AIPMTM
    # ========================================================

    elif (
        model_name
        ==
        "Proposed FA-AIPMTM (Full Model)"
    ):

        print(
            "\nUsing complete FA-AIPMTM feature configuration."
        )


        selected_indexes = (

            base_no_fuzzy_indexes

            + mfcc_indexes

            + fuzzy_indexes
        )


        selected_indexes = list(
            dict.fromkeys(
                selected_indexes
            )
        )


        Xtr = X_train_all[
            train_indices_sub
        ][:, :, selected_indexes]


        Xte = X_test[
            :,
            :,
            selected_indexes
        ]


        accuracy, f1, model = (
            train_ablation_model(

                Xtr,

                y_train_all[
                    train_indices_sub
                ],

                Xte,

                y_test,

                input_size=len(
                    selected_indexes
                ),

                hidden_size=HIDDEN_SIZE,

                num_layers=NUM_LAYERS,

                dropout=DROPOUT,

                learning_rate=LEARNING_RATE,

                batch_size=BATCH_SIZE,

                epochs=30
            )
        )


    # ========================================================
    # STORE RESULT
    # ========================================================

    result = config.copy()

    result[
        "Accuracy (%)"
    ] = round(
        accuracy * 100,
        2
    )

    result[
        "F1-Score (%)"
    ] = round(
        f1 * 100,
        2
    )


    results.append(
        result
    )


    print(
        "\nAccuracy:",
        f"{accuracy * 100:.2f}%"
    )

    print(
        "F1-Score:",
        f"{f1 * 100:.2f}%"
    )


# ============================================================
# 14. CREATE TABLE 8
# ============================================================

ablation_df = pd.DataFrame(
    results
)


# Reorder columns

ablation_df = ablation_df[
    [
        "Model Configuration",
        "Preprocessing",
        "MFCC",
        "FIS",
        "IntLSTM",
        "DTLBO Optimization",
        "Accuracy (%)",
        "F1-Score (%)"
    ]
]


# ============================================================
# 15. DISPLAY TABLE
# ============================================================

print("\n")
print("=" * 120)
print("TABLE 8. ABLATION STUDY OF THE PROPOSED FA-AIPMTM MODEL")
print("=" * 120)

print(
    ablation_df.to_string(
        index=False
    )
)


# ============================================================
# 16. SAVE CSV
# ============================================================

CSV_OUTPUT = os.path.join(
    OUTPUT_PATH,
    "Table_8_FA_AIPMTM_Ablation.csv"
)

ablation_df.to_csv(
    CSV_OUTPUT,
    index=False
)


# ============================================================
# 17. SAVE EXCEL
# ============================================================

EXCEL_OUTPUT = os.path.join(
    OUTPUT_PATH,
    "Table_8_FA_AIPMTM_Ablation.xlsx"
)

ablation_df.to_excel(
    EXCEL_OUTPUT,
    index=False
)


# ============================================================
# 18. SAVE LATEX TABLE
# ============================================================

LATEX_OUTPUT = os.path.join(
    OUTPUT_PATH,
    "Table_8_FA_AIPMTM_Ablation.tex"
)


with open(
    LATEX_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        ablation_df.to_latex(
            index=False,
            escape=False
        )
    )


# ============================================================
# 19. SAVE JSON
# ============================================================

JSON_OUTPUT = os.path.join(
    OUTPUT_PATH,
    "Table_8_FA_AIPMTM_Ablation.json"
)


ablation_df.to_json(
    JSON_OUTPUT,
    orient="records",
    indent=4
)


# ============================================================
# 20. FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 80)
print("TABLE 8 GENERATED SUCCESSFULLY")
print("=" * 80)

print(
    "\nCSV:",
    CSV_OUTPUT
)

print(
    "Excel:",
    EXCEL_OUTPUT
)

print(
    "LaTeX:",
    LATEX_OUTPUT
)

print(
    "JSON:",
    JSON_OUTPUT
)

print("\n" + "=" * 80)


Ablation Study
                                    Model Configuration Preprocessing MFCC FIS IntLSTM DTLBO Optimization  Accuracy (%)  F1-Score (%)
                                    LSTM Baseline Model             ✗    ✗   ✗       ✗                  ✗         89.86         89.22
                                               FIS Only             ✗    ✗   ✓       ✗                  ✗         90.85         90.42
IntLSTM + Preprocessing (Noise Removal + Normalization)             ✓    ✗   ✗       ✓                  ✗         91.34         90.77
                      IntLSTM + MFCC Feature Extraction             ✓    ✓   ✗       ✓                  ✗         92.47         91.99
                 IntLSTM + Fuzzy Inference System (FIS)             ✓    ✓   ✓       ✓                  ✗         94.10         92.85
                            DTL+IntLSTM Learning Module             ✓    ✓   ✗       ✓                  ✓         95.20         93.20
                        Proposed FA-AIPMTM (Fu

In [7]:
# ================================================================
# TABLE 9
# Cross-Validation Results of the Proposed FA-AIPMTM Model
# 5-Fold Stratified Cross-Validation, Seed = 42
# ================================================================

import os
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ------------------------------------------------
# 1. PATHS
# ------------------------------------------------

BASE_DIR = r"F:\.0 Work\4358\Data"

STEP6_DIR = os.path.join(BASE_DIR, "step6_output")
STEP8_DIR = os.path.join(BASE_DIR, "step8_output")

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "table9_cross_validation"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("TABLE 9: 5-FOLD STRATIFIED CROSS-VALIDATION")
print("=" * 70)

# ------------------------------------------------
# 2. RANDOM SEED
# ------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"\nRandom Seed: {SEED}")

# ------------------------------------------------
# 3. DEVICE
# ------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ------------------------------------------------
# 4. LOAD STEP 6 SEQUENCE DATA
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP 1: LOADING SEQUENCE DATA")
print("=" * 70)

X_train = np.load(
    os.path.join(
        STEP6_DIR,
        "X_train_sequences.npy"
    )
)

y_train = np.load(
    os.path.join(
        STEP6_DIR,
        "y_train_sequences.npy"
    )
)

X_test = np.load(
    os.path.join(
        STEP6_DIR,
        "X_test_sequences.npy"
    )
)

y_test = np.load(
    os.path.join(
        STEP6_DIR,
        "y_test_sequences.npy"
    )
)

print(f"\nTraining sequence shape : {X_train.shape}")
print(f"Training labels shape   : {y_train.shape}")
print(f"Test sequence shape     : {X_test.shape}")
print(f"Test labels shape       : {y_test.shape}")

# ------------------------------------------------
# 5. CONVERT TO NUMPY FLOAT32 / INT64
# ------------------------------------------------

X_train = X_train.astype(np.float32)
y_train = y_train.astype(np.int64)

X_test = X_test.astype(np.float32)
y_test = y_test.astype(np.int64)

# ------------------------------------------------
# 6. LOAD DTLBO BEST PARAMETERS
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2: LOADING DTLBO OPTIMIZED PARAMETERS")
print("=" * 70)

best_params_path = os.path.join(
    STEP8_DIR,
    "best_dtlbo_parameters.json"
)

if os.path.exists(best_params_path):

    with open(best_params_path, "r") as f:
        best_params = json.load(f)

    print("\nDTLBO optimized parameters:")
    print(json.dumps(best_params, indent=4))

else:

    print("\nWARNING:")
    print("DTLBO parameter file was not found.")
    print("Using manuscript-specified IntLSTM parameters.")

    best_params = {
        "hidden_size": 128,
        "num_layers": 2,
        "dropout": 0.30,
        "learning_rate": 0.001,
        "batch_size": 32
    }

# ------------------------------------------------
# 7. EXTRACT PARAMETERS SAFELY
# ------------------------------------------------

hidden_size = int(
    best_params.get("hidden_size", 128)
)

num_layers = int(
    best_params.get("num_layers", 2)
)

dropout = float(
    best_params.get("dropout", 0.30)
)

learning_rate = float(
    best_params.get("learning_rate", 0.001)
)

batch_size = int(
    best_params.get("batch_size", 32)
)

EPOCHS = 50

print("\nFinal parameters used for CV:")
print(f"Hidden Size    : {hidden_size}")
print(f"Layers         : {num_layers}")
print(f"Dropout        : {dropout}")
print(f"Learning Rate  : {learning_rate}")
print(f"Batch Size     : {batch_size}")
print(f"Epochs         : {EPOCHS}")

# ------------------------------------------------
# 8. NUMBER OF FEATURES / CLASSES
# ------------------------------------------------

SEQ_LENGTH = X_train.shape[1]
INPUT_SIZE = X_train.shape[2]
NUM_CLASSES = len(np.unique(y_train))

print("\nModel dimensions:")
print(f"Sequence Length : {SEQ_LENGTH}")
print(f"Input Features  : {INPUT_SIZE}")
print(f"Number Classes  : {NUM_CLASSES}")

# ------------------------------------------------
# 9. DEFINE DTL-INTLSTM MODEL
# ------------------------------------------------

class DTLIntLSTM(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        num_classes,
        dropout
    ):

        super(DTLIntLSTM, self).__init__()

        # PyTorch ignores dropout inside LSTM when
        # num_layers = 1, therefore handle it explicitly.
        lstm_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        # Final hidden state
        last_hidden = hidden[-1]

        x = self.dropout(last_hidden)

        x = self.classifier(x)

        return x


# ------------------------------------------------
# 10. TRAINING FUNCTION
# ------------------------------------------------

def train_one_fold(
    X_fold_train,
    y_fold_train,
    X_fold_val,
    y_fold_val,
    fold_number
):

    print("\n" + "-" * 70)
    print(f"TRAINING FOLD {fold_number}")
    print("-" * 70)

    # ------------------------------------------------
    # Convert to tensors
    # ------------------------------------------------

    Xtr = torch.tensor(
        X_fold_train,
        dtype=torch.float32
    )

    ytr = torch.tensor(
        y_fold_train,
        dtype=torch.long
    )

    Xval = torch.tensor(
        X_fold_val,
        dtype=torch.float32
    )

    yval = torch.tensor(
        y_fold_val,
        dtype=torch.long
    )

    # ------------------------------------------------
    # Dataset / DataLoader
    # ------------------------------------------------

    train_dataset = TensorDataset(
        Xtr,
        ytr
    )

    val_dataset = TensorDataset(
        Xval,
        yval
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    # ------------------------------------------------
    # Create model
    # ------------------------------------------------

    model = DTLIntLSTM(
        input_size=INPUT_SIZE,
        hidden_size=hidden_size,
        num_layers=num_layers,
        num_classes=NUM_CLASSES,
        dropout=dropout
    ).to(DEVICE)

    # ------------------------------------------------
    # Loss and optimizer
    # ------------------------------------------------

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    # ------------------------------------------------
    # Early stopping
    # ------------------------------------------------

    patience = 5
    patience_counter = 0

    best_val_loss = float("inf")

    best_state = None

    # ------------------------------------------------
    # Training
    # ------------------------------------------------

    for epoch in range(EPOCHS):

        model.train()

        train_loss = 0.0

        for batch_X, batch_y in train_loader:

            batch_X = batch_X.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(batch_X)

            loss = criterion(
                outputs,
                batch_y
            )

            loss.backward()

            optimizer.step()

            train_loss += (
                loss.item() *
                batch_X.size(0)
            )

        train_loss /= len(train_loader.dataset)

        # ------------------------------------------------
        # Validation
        # ------------------------------------------------

        model.eval()

        val_loss = 0.0

        with torch.no_grad():

            for batch_X, batch_y in val_loader:

                batch_X = batch_X.to(DEVICE)
                batch_y = batch_y.to(DEVICE)

                outputs = model(batch_X)

                loss = criterion(
                    outputs,
                    batch_y
                )

                val_loss += (
                    loss.item() *
                    batch_X.size(0)
                )

        val_loss /= len(val_loader.dataset)

        # ------------------------------------------------
        # Save best model
        # ------------------------------------------------

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

            patience_counter = 0

        else:

            patience_counter += 1

        if (epoch + 1) % 5 == 0:

            print(
                f"Epoch {epoch + 1:02d}/{EPOCHS} | "
                f"Train Loss: {train_loss:.5f} | "
                f"Val Loss: {val_loss:.5f}"
            )

        if patience_counter >= patience:

            print(
                f"Early stopping at epoch "
                f"{epoch + 1}"
            )

            break

    # ------------------------------------------------
    # Restore best model
    # ------------------------------------------------

    if best_state is not None:

        model.load_state_dict(
            best_state
        )

    # ------------------------------------------------
    # Validation prediction
    # ------------------------------------------------

    model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():

        for batch_X, batch_y in val_loader:

            batch_X = batch_X.to(DEVICE)

            outputs = model(batch_X)

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_targets.extend(
                batch_y.numpy()
            )

    all_predictions = np.array(
        all_predictions
    )

    all_targets = np.array(
        all_targets
    )

    # ------------------------------------------------
    # Metrics
    # ------------------------------------------------

    accuracy = accuracy_score(
        all_targets,
        all_predictions
    )

    precision = precision_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    )

    print(
        f"\nFold {fold_number} Results:"
    )

    print(
        f"Accuracy  : {accuracy * 100:.4f}%"
    )

    print(
        f"Precision : {precision * 100:.4f}%"
    )

    print(
        f"Recall    : {recall * 100:.4f}%"
    )

    print(
        f"F1-Score  : {f1 * 100:.4f}%"
    )

    return {
        "Fold": fold_number,
        "Accuracy (%)": accuracy * 100,
        "Precision (%)": precision * 100,
        "Recall (%)": recall * 100,
        "F1-Score (%)": f1 * 100
    }


# ================================================================
# 11. 5-FOLD STRATIFIED CROSS-VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("STEP 3: 5-FOLD STRATIFIED CROSS-VALIDATION")
print("=" * 70)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

cv_results = []

# ------------------------------------------------
# IMPORTANT:
# CV is performed ONLY on the training sequences.
# The independent test set remains untouched.
# ------------------------------------------------

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train, y_train),
    start=1
):

    X_fold_train = X_train[
        train_idx
    ]

    y_fold_train = y_train[
        train_idx
    ]

    X_fold_val = X_train[
        val_idx
    ]

    y_fold_val = y_train[
        val_idx
    ]

    print(
        f"\nFold {fold}: "
        f"Training = {len(train_idx)}, "
        f"Validation = {len(val_idx)}"
    )

    fold_result = train_one_fold(
        X_fold_train,
        y_fold_train,
        X_fold_val,
        y_fold_val,
        fold
    )

    cv_results.append(
        fold_result
    )


# ================================================================
# 12. CREATE FOLD RESULTS DATAFRAME
# ================================================================

results_df = pd.DataFrame(
    cv_results
)

print("\n" + "=" * 70)
print("FOLD-WISE RESULTS")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ================================================================
# 13. CALCULATE MEAN ± SD
# ================================================================

metric_columns = [
    "Accuracy (%)",
    "Precision (%)",
    "Recall (%)",
    "F1-Score (%)"
]

mean_values = results_df[
    metric_columns
].mean()

std_values = results_df[
    metric_columns
].std(
    ddof=1
)

print("\n" + "=" * 70)
print("MEAN ± SD")
print("=" * 70)

for metric in metric_columns:

    print(
        f"{metric}: "
        f"{mean_values[metric]:.4f} ± "
        f"{std_values[metric]:.4f}"
    )


# ================================================================
# 14. 95% CONFIDENCE INTERVAL
# ================================================================
#
# Since n = 5 folds, use the t-distribution rather than
# the normal approximation.
#
# CI = mean ± t_(0.975, 4) * SD / sqrt(5)
#
# t critical value for df=4 ≈ 2.776445
# ================================================================

N_FOLDS = 5

T_CRITICAL = 2.7764451051977987

ci_results = {}

for metric in metric_columns:

    mean = mean_values[metric]

    sd = std_values[metric]

    standard_error = (
        sd /
        np.sqrt(N_FOLDS)
    )

    margin = (
        T_CRITICAL *
        standard_error
    )

    lower = mean - margin
    upper = mean + margin

    ci_results[metric] = {
        "Mean": mean,
        "SD": sd,
        "Lower_95_CI": lower,
        "Upper_95_CI": upper
    }

print("\n" + "=" * 70)
print("95% CONFIDENCE INTERVAL")
print("=" * 70)

for metric, values in ci_results.items():

    print(
        f"{metric}: "
        f"{values['Lower_95_CI']:.4f} - "
        f"{values['Upper_95_CI']:.4f}"
    )


# ================================================================
# 15. CREATE PUBLICATION-READY TABLE 9
# ================================================================

table9_rows = []

# ------------------------------------------------
# Fold 1–5
# ------------------------------------------------

for _, row in results_df.iterrows():

    table9_rows.append({
        "Fold": f"Fold {int(row['Fold'])}",
        "Accuracy (%)": f"{row['Accuracy (%)']:.2f}",
        "Precision (%)": f"{row['Precision (%)']:.2f}",
        "Recall (%)": f"{row['Recall (%)']:.2f}",
        "F1-Score (%)": f"{row['F1-Score (%)']:.2f}"
    })


# ------------------------------------------------
# Mean ± SD
# ------------------------------------------------

table9_rows.append({
    "Fold": "Mean ± SD",
    "Accuracy (%)":
        f"{mean_values['Accuracy (%)']:.2f} ± "
        f"{std_values['Accuracy (%)']:.2f}",

    "Precision (%)":
        f"{mean_values['Precision (%)']:.2f} ± "
        f"{std_values['Precision (%)']:.2f}",

    "Recall (%)":
        f"{mean_values['Recall (%)']:.2f} ± "
        f"{std_values['Recall (%)']:.2f}",

    "F1-Score (%)":
        f"{mean_values['F1-Score (%)']:.2f} ± "
        f"{std_values['F1-Score (%)']:.2f}"
})


# ------------------------------------------------
# 95% CI
# ------------------------------------------------

table9_rows.append({
    "Fold": "95% CI",

    "Accuracy (%)":
        f"{ci_results['Accuracy (%)']['Lower_95_CI']:.2f} – "
        f"{ci_results['Accuracy (%)']['Upper_95_CI']:.2f}",

    "Precision (%)":
        f"{ci_results['Precision (%)']['Lower_95_CI']:.2f} – "
        f"{ci_results['Precision (%)']['Upper_95_CI']:.2f}",

    "Recall (%)":
        f"{ci_results['Recall (%)']['Lower_95_CI']:.2f} – "
        f"{ci_results['Recall (%)']['Upper_95_CI']:.2f}",

    "F1-Score (%)":
        f"{ci_results['F1-Score (%)']['Lower_95_CI']:.2f} – "
        f"{ci_results['F1-Score (%)']['Upper_95_CI']:.2f}"
})

table9_df = pd.DataFrame(
    table9_rows
)


# ================================================================
# 16. DISPLAY FINAL TABLE
# ================================================================

print("\n" + "=" * 70)
print("TABLE 9: FINAL PUBLICATION TABLE")
print("=" * 70)

print(
    table9_df.to_string(
        index=False
    )
)


# ================================================================
# 17. SAVE CSV
# ================================================================

csv_path = os.path.join(
    OUTPUT_DIR,
    "Table_9_Cross_Validation_Results.csv"
)

table9_df.to_csv(
    csv_path,
    index=False
)

print(
    f"\nCSV saved to:\n{csv_path}"
)


# ================================================================
# 18. SAVE EXCEL
# ================================================================

excel_path = os.path.join(
    OUTPUT_DIR,
    "Table_9_Cross_Validation_Results.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    table9_df.to_excel(
        writer,
        sheet_name="Table 9",
        index=False
    )

    results_df.to_excel(
        writer,
        sheet_name="Fold Results",
        index=False
    )

print(
    f"Excel saved to:\n{excel_path}"
)


# ================================================================
# 19. SAVE RAW NUMERIC RESULTS
# ================================================================

raw_path = os.path.join(
    OUTPUT_DIR,
    "Table_9_Raw_Fold_Metrics.csv"
)

results_df.to_csv(
    raw_path,
    index=False
)


# ================================================================
# 20. SAVE STATISTICAL SUMMARY
# ================================================================

summary_records = []

for metric in metric_columns:

    summary_records.append({
        "Metric": metric,
        "Mean (%)":
            mean_values[metric],
        "SD (%)":
            std_values[metric],
        "95% CI Lower (%)":
            ci_results[metric]["Lower_95_CI"],
        "95% CI Upper (%)":
            ci_results[metric]["Upper_95_CI"]
    })

summary_df = pd.DataFrame(
    summary_records
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "Table_9_Statistical_Summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)


# ================================================================
# 21. SAVE JSON
# ================================================================

json_output = {
    "seed": SEED,
    "n_folds": 5,
    "cv_method": "StratifiedKFold",
    "shuffle": True,
    "test_set_used_during_cv": False,
    "model": "FA-AIPMTM",
    "model_architecture": "DTL-IntLSTM",
    "parameters": {
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "epochs": EPOCHS,
        "early_stopping_patience": 5
    },
    "fold_results":
        results_df.to_dict(
            orient="records"
        ),
    "summary":
        summary_records
}

json_path = os.path.join(
    OUTPUT_DIR,
    "Table_9_CV_Statistics.json"
)

with open(
    json_path,
    "w"
) as f:

    json.dump(
        json_output,
        f,
        indent=4
    )


# ================================================================
# 22. FINAL MESSAGE
# ================================================================

print("\n" + "=" * 70)
print("TABLE 9 GENERATION COMPLETED")
print("=" * 70)

print("\nGenerated files:")

print(
    f"1. {csv_path}"
)

print(
    f"2. {excel_path}"
)

print(
    f"3. {raw_path}"
)

print(
    f"4. {summary_path}"
)

print(
    f"5. {json_path}"
)

print("\nFinal Table 9:")
print(
    table9_df.to_string(
        index=False
    )
)


Cross-Validation Results
(5-Fold Stratified, Seed=42)
     Fold   Accuracy (%)  Precision (%)     Recall (%)   F1-Score (%)
   Fold 1          96.40          96.55          96.30          96.42
   Fold 2          96.62          96.70          96.48          96.59
   Fold 3          96.78          96.85          96.60          96.72
   Fold 4          96.55          96.60          96.35          96.47
   Fold 5          96.70          96.78          96.52          96.65
Mean ± SD   96.61 ± 0.15   96.70 ± 0.12   96.45 ± 0.12   96.57 ± 0.12
   95% CI [96.42, 96.80] [96.55, 96.85] [96.30, 96.60] [96.42, 96.72]
